# Radar simulator v2.1: generate the data, save the CSV, explore it (interactive, dark mode)
This is a single self-contained notebook (the simulator is inside it). Run the cells from top to bottom. `pip install plotly pandas numpy` if needed.

| Cell | What it does |
|---|---|
| 1 | Settings |
| 2 | **Generates the tracks and saves the CSV** (plus the behaviour glossary CSV) |
| 3 | Loads the CSV, adds a per-packet motion label |
| 4 to 15 | Interactive plots (hover, zoom, click legend entries to hide/show classes) |
| 16 | Writes one self-contained **HTML report** (all plots + glossary + explanations) |

Key facts: noise-free truth on an exact 2.5 s grid, every class has (almost) the same number of packets, no train/val/test split column yet.
The missile classes are generic powered-flight profiles only (no guidance, homing or target logic).

## Part A: the simulator (just run these cells once; you normally do not need to edit them)

In [ ]:
# ---------------- Generator part 1: imports, constants and the motion engine ----------------
"""
radar_sim_v2.py  (v2.1)  -  physics-based, noise-free target simulator (2.5 s output grid)

Every target is a point mass flown by a simple autopilot through a *mission* (a chain of behaviours).
Dynamics run on a fine internal step; the radar-style packets are sampled exactly every 2.5 s: t = 0, 2.5, 5, ...
No measurement noise and no tracker in this version: position, velocity and acceleration are the TRUE values.

Two motion models
  'fw'  : velocity = (speed V, heading psi, flight-path angle gamma). Controls: speed, turn rate (bank-limited), climb.
          Used for aeroplanes, fixed-wing drones, missiles, boats and ships (ships/boats have gamma = 0).
  'vec' : ground-velocity vector with an acceleration (tilt) limit. Used for helicopters and multirotor drones,
          so they can stop, hover, translate sideways and land.
Axes: x = east, y = north, z = up (metres), radar at the origin.  heading = atan2(vy, vx) in degrees (as in the old file).
"""
import math, random
import numpy as np, pandas as pd

G = 9.81
DT_OUT = 2.5
RADAR_H = 20.0
CLASSES = ["CIVILIAN_AIRCRAFT", "FIGHTER_AIRCRAFT", "HELICOPTER", "DRONE", "SURFACE_MISSILE",
           "AERIAL_MISSILE", "SKIFF", "BOAT", "SHIP"]
ICD_COLS = ["packet_size", "source", "destination", "packet_type", "packet_category", "radar_id", "track_no",
            "pos_x", "pos_y", "pos_z", "range_mag", "vel_x", "vel_y", "vel_z", "vel_mag", "heading",
            "accel_x", "accel_y", "accel_z", "accel_mag", "std_dev_x", "std_dev_y", "std_dev_z",
            "std_dev_vx", "std_dev_vy", "std_dev_vz", "std_dev_ax", "std_dev_ay", "std_dev_az",
            "cov_x_vx", "cov_y_vy", "cov_z_vz", "ntp_sec", "ntp_frac", "assoc_indication"]

def wrap(a): return (a + math.pi) % (2 * math.pi) - math.pi
def clip(x, lo, hi): return lo if x < lo else (hi if x > hi else x)
def horizon(z): return 4120.0 * (math.sqrt(RADAR_H) + math.sqrt(max(z, 0.0)))


class Agent:
    def __init__(s, kind, rng, dt, **kw):
        s.kind, s.rng, s.dt = kind, rng, dt
        # state
        s.x = s.y = s.z = 0.0; s.V = 100.0; s.psi = 0.0; s.gam = 0.0
        s.vx = s.vy = s.vz = 0.0                                   # (vec model) ground velocity
        s.al = s.om = s.gd = 0.0; s.ax = s.ay = s.az = 0.0
        s.hdg = 0.0; s.t = 0.0
        # commands
        s.Vc = 100.0; s.psic = 0.0; s.zc = 0.0
        s.al_cmd = s.alat_cmd = s.avert_cmd = None                  # explicit accelerations (missiles)
        s.vcx = s.vcy = 0.0
        # limits / time constants (defaults = airliner-like); overridden by kw
        s.Vmin, s.Vmax = 40.0, 300.0; s.a_acc, s.a_dec = 1.0, 1.5; s.tau_v = 8.0; s.tau_a = 2.0
        s.glat = 0.45; s.om_cap = None; s.tau_om = 3.0; s.tau_psi = 6.0
        s.gam_max = math.radians(25); s.vz_max = 15.0; s.az_max = 1.5; s.tau_g = 4.0; s.tau_pitch = 2.0
        s.planar = False; s.zmin = 0.0; s.drag_k = 0.0; s.grav = True; s.n_lat_g = 0.0
        s.a_max = 3.0; s.tau_vz = 1.5                                 # vec model
        s.wind = (0.0, 0.0); s.wind_ref = 1.0
        s.heave = []; s.z_ref = 0.0
        s.phase = "init"; s.acting = None; s.done = False
        for k, v in kw.items(): setattr(s, k, v)

    # ---------- helpers ----------
    def turn_radius(self):
        om = (self.om_cap if self.om_cap else self.glat * G / max(self.V * math.cos(self.gam), 10.0))
        return max(self.V * math.cos(self.gam), 1.0) / max(om, 1e-6)

    def wind_at(self):
        f = clip(0.25 + self.z / 4000.0, 0.25, 1.0) * self.wind_ref
        return self.wind[0] * f, self.wind[1] * f

    # ---------- dynamics: fixed-wing / missile / ship ----------
    def step_fw(s):
        dt = s.dt; V = max(s.V, 1.0); cg = max(math.cos(s.gam), 0.1)
        if s.al_cmd is not None: al_t = s.al_cmd
        else: al_t = clip((s.Vc - s.V) / s.tau_v, -s.a_dec, s.a_acc)
        if s.grav and not s.planar: al_t -= G * math.sin(s.gam)
        al_t -= s.drag_k * s.n_lat_g ** 2
        s.al += (al_t - s.al) * min(1.0, dt / s.tau_a)
        omax = s.om_cap if s.om_cap else s.glat * G / (V * cg)
        if s.alat_cmd is not None: om_t = clip(s.alat_cmd / (V * cg), -omax, omax)
        else: om_t = clip(wrap(s.psic - s.psi) / s.tau_psi, -omax, omax)
        s.om += (om_t - s.om) * min(1.0, dt / s.tau_om)
        if s.planar: gd_t = 0.0
        elif s.avert_cmd is not None: gd_t = clip(s.avert_cmd / V, -s.az_max / V, s.az_max / V)
        else:
            e = s.zc - s.z
            vz_des = math.copysign(min(s.vz_max, math.sqrt(2 * 0.4 * s.az_max * abs(e)), abs(e) / 6.0 + 0.5), e)
            gam_des = math.asin(clip(vz_des / V, -math.sin(s.gam_max), math.sin(s.gam_max)))
            gd_t = clip((gam_des - s.gam) / s.tau_g, -s.az_max / V, s.az_max / V)
        s.gd += (gd_t - s.gd) * min(1.0, dt / s.tau_pitch)
        s.n_lat_g = math.hypot(V * s.om * cg, V * s.gd) / G
        Vo, po, go = s.V, s.psi, s.gam
        s.V = clip(s.V + s.al * dt, s.Vmin, s.Vmax); s.psi = wrap(s.psi + s.om * dt)
        s.gam = clip(s.gam + s.gd * dt, -math.radians(85), math.radians(85))
        wx, wy = s.wind_at()
        v0 = (Vo * math.cos(go) * math.cos(po), Vo * math.cos(go) * math.sin(po), Vo * math.sin(go))
        v1 = (s.V * math.cos(s.gam) * math.cos(s.psi), s.V * math.cos(s.gam) * math.sin(s.psi), s.V * math.sin(s.gam))
        s.x += 0.5 * (v0[0] + v1[0]) * dt + wx * dt; s.y += 0.5 * (v0[1] + v1[1]) * dt + wy * dt
        s.z += 0.5 * (v0[2] + v1[2]) * dt
        if s.z < s.zmin:
            s.z = s.zmin
            if s.gam < 0: s.gam = 0.0; s.gd = 0.0
        s.t += dt

    def out_fw(s):
        cg, sg, cp, sp = math.cos(s.gam), math.sin(s.gam), math.cos(s.psi), math.sin(s.psi)
        wx, wy = s.wind_at()
        v = (s.V * cg * cp + wx, s.V * cg * sp + wy, s.V * sg)
        a = (s.al * cg * cp - s.V * s.gd * sg * cp - s.V * cg * sp * s.om,
             s.al * cg * sp - s.V * s.gd * sg * sp + s.V * cg * cp * s.om,
             s.al * sg + s.V * s.gd * cg)
        return v, a

    # ---------- dynamics: rotorcraft / multirotor ----------
    def step_vec(s):
        dt = s.dt
        ax_t = (s.vcx - s.vx) / s.tau_v; ay_t = (s.vcy - s.vy) / s.tau_v
        m = math.hypot(ax_t, ay_t)
        if m > s.a_max: ax_t *= s.a_max / m; ay_t *= s.a_max / m
        k = min(1.0, dt / s.tau_a)
        s.ax += (ax_t - s.ax) * k; s.ay += (ay_t - s.ay) * k
        e = s.zc - s.z
        vzc = math.copysign(min(s.vz_max, math.sqrt(2 * 0.5 * s.az_max * abs(e)), abs(e) / 3.0), e)
        az_t = clip((vzc - s.vz) / s.tau_vz, -s.az_max, s.az_max)
        s.az += (az_t - s.az) * k
        vx0, vy0, vz0 = s.vx, s.vy, s.vz
        s.vx += s.ax * dt; s.vy += s.ay * dt; s.vz += s.az * dt
        s.x += 0.5 * (vx0 + s.vx) * dt; s.y += 0.5 * (vy0 + s.vy) * dt; s.z += 0.5 * (vz0 + s.vz) * dt
        if s.z < s.zmin: s.z = s.zmin; s.vz = max(s.vz, 0.0); s.az = max(s.az, 0.0)
        s.t += dt

    def out_vec(s):
        return (s.vx, s.vy, s.vz), (s.ax, s.ay, s.az)

    def step(s):
        s.step_fw() if s.kind == "fw" else s.step_vec()

    def sample(s):
        v, a = s.out_fw() if s.kind == "fw" else s.out_vec()
        x, y, z = s.x, s.y, s.z
        if s.heave:                                          # deterministic sea heave (surface targets)
            for A, T, ph in s.heave:
                w = 2 * math.pi / T
                z += A * math.sin(w * s.t + ph); v = (v[0], v[1], v[2] + A * w * math.cos(w * s.t + ph))
                a = (a[0], a[1], a[2] - A * w * w * math.sin(w * s.t + ph))
        hs = math.hypot(v[0], v[1])
        if hs > 0.5: s.hdg = math.atan2(v[1], v[0])
        return x, y, z, v, a, s.hdg

In [ ]:
# ---------------- Generator part 2: behaviour building blocks ----------------
# =========================================================================================
#  behaviour building blocks (generators: each `yield` advances one fine time step)
# =========================================================================================
def hold(ag, T):
    for _ in range(int(T / ag.dt)): yield

def fw_goto(ag, wx, wy, V, z, tol=400.0, Tmax=1e9):
    tol = max(tol, 1.3 * ag.turn_radius()); t = 0.0
    yield
    while t < Tmax:
        dx, dy = wx - ag.x, wy - ag.y; d = math.hypot(dx, dy)
        if d < tol: return
        ag.psic = math.atan2(dy, dx); ag.Vc = V; ag.zc = z
        yield; t += ag.dt

def fw_leg(ag, psi, T, V, z):
    ag.psic = psi; ag.Vc = V; ag.zc = z
    for _ in range(int(T / ag.dt)): yield

def fw_alt(ag, z, V, T=1e9, tol=40.0):
    ag.psic = ag.psi; ag.Vc = V; ag.zc = z; t = 0.0
    while abs(ag.z - z) > tol and t < T: yield; t += ag.dt

def orbit_dir(x, y, cx, cy, R, ccw):
    rx, ry = x - cx, y - cy; d = max(math.hypot(rx, ry), 1.0); rx /= d; ry /= d
    tx, ty = (-ry, rx) if ccw else (ry, -rx)
    kap = clip((d - R) / (0.6 * R), -1.0, 1.0) * 1.1
    ux = tx * math.cos(kap) - rx * math.sin(kap); uy = ty * math.cos(kap) - ry * math.sin(kap)
    return ux, uy

def fw_orbit(ag, cx, cy, R, ccw, V, z, T):
    R = max(R, 1.15 * ag.turn_radius())
    for _ in range(int(T / ag.dt)):
        ux, uy = orbit_dir(ag.x, ag.y, cx, cy, R, ccw)
        ag.psic = math.atan2(uy, ux); ag.Vc = V; ag.zc = z
        yield

def fw_racetrack(ag, hdg, L, V, z, T, right=True, tol=300.0):
    R = 1.2 * ag.turn_radius(); sgn = -1 if right else 1
    ex, ey = math.cos(hdg), math.sin(hdg); nx, ny = -ey * sgn, ex * sgn
    x0, y0 = ag.x, ag.y; t0 = ag.t
    pts = [(x0 + ex * L, y0 + ey * L), (x0 + ex * L + 2 * R * nx, y0 + ey * L + 2 * R * ny),
           (x0 + 2 * R * nx, y0 + 2 * R * ny), (x0, y0)]
    i = 0
    while ag.t - t0 < T:
        yield from fw_goto(ag, pts[i % 4][0], pts[i % 4][1], V, z, tol=tol, Tmax=T - (ag.t - t0)); i += 1

def vec_set_dir(ag, ux, uy, speed): ag.vcx, ag.vcy = ux * speed, uy * speed

def vec_goto(ag, wx, wy, speed, z, tol=4.0, stop=True, Tmax=1e9):
    t = 0.0
    while t < Tmax:
        dx, dy = wx - ag.x, wy - ag.y; d = math.hypot(dx, dy)
        if stop:
            if d < tol and math.hypot(ag.vx, ag.vy) < 0.8: return
            v = min(speed, math.sqrt(2 * ag.a_max * 0.5 * max(d - 0.5 * tol, 0.0)))
        else:
            if d < tol: return
            v = speed
        vec_set_dir(ag, dx / max(d, 1e-6), dy / max(d, 1e-6), v); ag.zc = z
        yield; t += ag.dt

def vec_orbit(ag, cx, cy, R, ccw, speed, z, T):
    for _ in range(int(T / ag.dt)):
        ux, uy = orbit_dir(ag.x, ag.y, cx, cy, R, ccw)
        vec_set_dir(ag, ux, uy, speed); ag.zc = z
        yield

def vec_hover(ag, T, amp=None, alt_amp=0.3):
    """Station keeping: hold a point with a small, slow, deterministic wander (gust response / rotor wash)."""
    r = ag.rng; amp = amp if amp is not None else r.uniform(0.15, 0.7)
    comps = [(amp * r.uniform(0.5, 1.0), r.uniform(0.05, 0.45), r.uniform(0, 6.28)) for _ in range(3)]
    compsy = [(amp * r.uniform(0.5, 1.0), r.uniform(0.05, 0.45), r.uniform(0, 6.28)) for _ in range(3)]
    hx, hy, hz = ag.x, ag.y, ag.zc if ag.zc > 0 else ag.z; ph = r.uniform(0, 6.28); t0 = ag.t
    for _ in range(int(T / ag.dt)):
        tt = ag.t - t0
        wx = sum(a * math.sin(w * tt + p) for a, w, p in comps) / 2.0
        wy = sum(a * math.sin(w * tt + p) for a, w, p in compsy) / 2.0
        ag.vcx = clip(0.4 * (hx - ag.x), -2, 2) + wx; ag.vcy = clip(0.4 * (hy - ag.y), -2, 2) + wy
        ag.zc = hz + alt_amp * math.sin(0.3 * tt + ph)
        yield

def vec_vertical(ag, z, rate, hold_xy=True):
    ag.vz_max = rate; hx, hy = ag.x, ag.y
    while abs(ag.z - z) > 1.5:
        if hold_xy: ag.vcx = clip(0.4 * (hx - ag.x), -2, 2); ag.vcy = clip(0.4 * (hy - ag.y), -2, 2)
        ag.zc = z; yield

In [ ]:
# ---------------- Generator part 3: one behaviour recipe per class ----------------
# =========================================================================================
#  class factories: each returns (agent, mission generator, meta dict)
# =========================================================================================
rad = math.radians

def place(ag, rng, r_lo, r_hi, mode=None):
    r0 = rng.uniform(r_lo, max(r_hi, r_lo + 1000.0)); th = rng.uniform(0, 2 * math.pi)
    ag.x, ag.y = r0 * math.cos(th), r0 * math.sin(th)
    mode = mode or rng.choice(["inbound", "crossing", "outbound", "random"])
    base = math.atan2(-ag.y, -ag.x)
    off = {"inbound": rng.gauss(0, 0.25), "outbound": math.pi + rng.gauss(0, 0.25),
           "crossing": rng.choice([-1, 1]) * (math.pi / 2 + rng.gauss(0, 0.3)), "random": rng.uniform(-math.pi, math.pi)}[mode]
    return wrap(base + off), mode

def set_air_wind(ag, rng, vmax):
    sp = rng.uniform(0, vmax); d = rng.uniform(0, 2 * math.pi); ag.wind = (sp * math.cos(d), sp * math.sin(d))

def fw_init(ag, psi, V, z):
    ag.psi = ag.psic = psi; ag.V = ag.Vc = V; ag.z = ag.zc = z; ag.gam = 0.0; ag.hdg = psi

def vec_init(ag, psi, V, z):
    ag.vx, ag.vy = V * math.cos(psi), V * math.sin(psi); ag.vcx, ag.vcy = ag.vx, ag.vy; ag.z = ag.zc = z; ag.hdg = psi

def next_wp(ag, rng, L, dmax_deg, dpref=22.0):
    d = clip(rng.gauss(0, rad(dpref)), -rad(dmax_deg), rad(dmax_deg))
    if rng.random() < 0.35: d = rng.choice([-1, 1]) * rng.uniform(0.4, rad(dmax_deg))
    ref = ag.psi if ag.kind == "fw" else ag.hdg
    return ag.x + L * math.cos(ref + d), ag.y + L * math.sin(ref + d)

def vary(rng, base, frac): return base * rng.uniform(1 - frac, 1 + frac)

def ramp_speed(ag, rng, V_to, style):
    """Breakout speed-up: abrupt (one step) or gradual (several steps)."""
    if style == "gradual":
        V0 = ag.Vc; n = rng.randint(3, 6)
        for k in range(1, n + 1):
            ag.Vc = V0 + (V_to - V0) * k / n; yield from hold(ag, rng.uniform(12, 30))
    else:
        ag.Vc = V_to; yield from hold(ag, 4)

# ---------------------------------------------------------------- CIVILIAN AIRCRAFT
def make_civilian(rng, dt, p_disguise=0.0):
    sub = rng.choices(["airliner", "turboprop", "light_ga"], [0.55, 0.2, 0.25])[0]
    if sub == "airliner":
        P = dict(Vmin=70, Vmax=300, a_acc=0.8, a_dec=1.2, glat=rng.uniform(0.30, 0.50), vz_max=rng.uniform(8, 15), az_max=1.0,
                 tau_om=3.0, tau_psi=8.0, tau_g=5.0, tau_pitch=2.5, zmin=30, gam_max=rad(12))
        cruise, Vc, zlo, zhi, dzr = rng.uniform(9000, 12500), rng.uniform(205, 255), 3000, 12500, (300, 1500)
    elif sub == "turboprop":
        P = dict(Vmin=55, Vmax=170, a_acc=0.7, a_dec=1.0, glat=rng.uniform(0.30, 0.45), vz_max=rng.uniform(5, 10), az_max=1.0,
                 tau_om=2.5, tau_psi=6.0, tau_g=4.0, tau_pitch=2.0, zmin=30, gam_max=rad(12))
        cruise, Vc, zlo, zhi, dzr = rng.uniform(3000, 7000), rng.uniform(105, 150), 1500, 7500, (200, 1000)
    else:
        P = dict(Vmin=28, Vmax=100, a_acc=0.8, a_dec=1.0, glat=rng.uniform(0.35, 0.55), vz_max=rng.uniform(2, 5), az_max=0.8,
                 tau_om=1.5, tau_psi=4.0, tau_g=3.0, tau_pitch=1.5, zmin=20, gam_max=rad(10))
        cruise, Vc, zlo, zhi, dzr = rng.uniform(300, 3500), rng.uniform(45, 75), 200, 4000, (100, 600)
    ag = Agent("fw", rng, dt, **P); ag.acting = "CIVILIAN_AIRCRAFT"
    set_air_wind(ag, rng, 30 if sub == "airliner" else 15)
    prof = rng.choices(["enroute", "climb", "descent", "hold"], [0.35, 0.2, 0.25, 0.2])[0]
    psi, mode = place(ag, rng, 15e3, min(140e3, horizon(cruise) * 0.9))
    z0 = cruise if prof in ("enroute", "descent") else (rng.uniform(600, 2500) * (1 if sub == "airliner" else 0.5) if prof == "climb" else cruise * rng.uniform(0.3, 0.6))
    fw_init(ag, psi, Vc if prof != "climb" else Vc * 0.6, z0)
    Lsc = 1.0 if sub == "airliner" else (0.7 if sub == "turboprop" else 0.2)
    def mission():
        if prof == "climb": ag.phase = "climb"; yield from fw_alt(ag, cruise, Vc, tol=200)
        elif prof == "descent":
            ag.phase = "descent"; yield from fw_alt(ag, cruise * 0.35, Vc * 0.8, tol=150); yield from fw_alt(ag, cruise * 0.15, Vc * 0.55, tol=100)
            ag.phase = "approach"; ag.vz_max = 4.0; yield from fw_alt(ag, 60.0, Vc * 0.40, tol=20)
            ag.phase = "final"; yield from fw_leg(ag, ag.psi, 1e9, Vc * 0.38, 30)
        elif prof == "hold":
            ag.phase = "holding"; yield from fw_racetrack(ag, ag.psi, Vc * 0.6 * 50.0, Vc * 0.6, ag.z, rng.uniform(250, 500), right=rng.random() < 0.5)
        while True:
            r = rng.random(); Vn = clip(vary(rng, Vc, 0.12), P["Vmin"] * 1.5, P["Vmax"] * 0.9)
            if r < 0.50:
                L = rng.uniform(6e3, 40e3) * Lsc + 2e3; tx, ty = next_wp(ag, rng, L, 100, 28); zt = ag.z; ag.phase = "transit"
                if rng.random() < 0.3:
                    dz = rng.choice([-1, 1]) * rng.uniform(*dzr); zt = clip(ag.z + dz, zlo, zhi); dz = zt - ag.z
                    ag.phase = "climb" if dz > 150 else ("descent" if dz < -150 else "transit")
                yield from fw_goto(ag, tx, ty, Vn, zt)
            elif r < 0.62:
                ag.phase = "weather_deviation"; g0 = ag.glat; ag.glat = min(g0 * 1.5, 0.7); psi0 = ag.psi; ang = rng.choice([-1, 1]) * rng.uniform(0.4, 0.8)
                yield from fw_leg(ag, psi0 + ang, rng.uniform(40, 100), Vn, ag.z); yield from fw_leg(ag, psi0, rng.uniform(40, 100), Vn, ag.z); ag.glat = g0
            elif r < 0.74:
                ag.phase = "holding"; yield from fw_racetrack(ag, ag.psi, Vn * 0.6 * 50.0, Vn * 0.6, ag.z, rng.uniform(250, 500), right=rng.random() < 0.5)
            elif r < 0.86:
                ag.phase = "orbit"; R = rng.uniform(2e3, 9e3) * (1 if sub == "airliner" else 0.4); c = (ag.x + R * math.cos(ag.psi + 1.57), ag.y + R * math.sin(ag.psi + 1.57))
                yield from fw_orbit(ag, c[0], c[1], R, rng.random() < 0.5, Vn, ag.z, rng.uniform(100, 300))
            else:
                ag.phase = "speed_change"; Vn2 = clip(Vc * rng.uniform(0.75, 1.15), P["Vmin"] * 1.5, P["Vmax"] * 0.9)
                yield from fw_leg(ag, ag.psi, rng.uniform(60, 150), Vn2, ag.z)
    return ag, mission(), dict(subtype=sub, profile=prof, geometry=mode)

# ---------------------------------------------------------------- FIGHTER AIRCRAFT
def make_fighter(rng, dt, p_disguise=0.45):
    disguised = rng.random() < p_disguise
    sub = "high_speed" if disguised else rng.choices(["cap", "high_speed", "bfm", "low_level", "climb_cycles", "transit"], [0.15, 0.2, 0.25, 0.15, 0.15, 0.10])[0]
    P = dict(Vmin=110, Vmax=900, a_acc=6.0, a_dec=8.0, glat=3.0, vz_max=180, az_max=6 * G, tau_om=0.9, tau_psi=2.0, tau_a=1.5,
             tau_g=1.5, tau_pitch=0.9, tau_v=6.0, zmin=15, gam_max=rad(65), drag_k=0.07)
    ag = Agent("fw", rng, dt, **P); ag.acting = "FIGHTER_AIRCRAFT"; set_air_wind(ag, rng, 35)
    zc = rng.uniform(7000, 12000) if sub not in ("low_level", "climb_cycles") else rng.uniform(80, 300)
    Vc = rng.uniform(200, 260) if sub in ("cap", "transit") else rng.uniform(230, 300)
    psi, mode = place(ag, rng, 15e3, min(140e3, horizon(zc) * 0.9)); fw_init(ag, psi, Vc, zc)
    style = rng.choice(["abrupt", "gradual"]) if disguised else ""
    def hard_turn(n, dpsi, Vtarget):
        ag.glat = n; ag.psic = wrap(ag.psi + dpsi); ag.Vc = Vtarget; ag.phase = "hard_turn"; t = 0.0
        while abs(wrap(ag.psic - ag.psi)) > 0.12 and t < 40: yield; t += ag.dt
        ag.glat = 3.0
    def mission():
        if sub == "cap":
            ag.phase = "cap_racetrack"; Lr = rng.uniform(30e3, 60e3)
            while True:
                ag.phase = "cap_racetrack"; yield from fw_racetrack(ag, ag.psi, Lr, vary(rng, Vc, .1), zc, rng.uniform(200, 600), right=rng.random() < 0.5)
                ag.phase = "sprint"; yield from fw_leg(ag, ag.psi, rng.uniform(20, 50), rng.uniform(380, 520), zc)
        elif sub == "high_speed":
            if disguised:
                ag.acting = "CIVILIAN_AIRCRAFT"; ag.phase = "mimic_civilian"; ag.glat = 0.4; ag.a_acc = 1.0; ag.drag_k = 0.0
                ag.Vc = rng.uniform(210, 245); ag.zc = rng.uniform(9000, 11000); ag.vz_max = 12; ag.V = ag.Vc; ag.z = ag.zc
                T0 = ag.t; Tm = rng.uniform(150, 400)
                while ag.t - T0 < Tm:
                    r = rng.random()
                    if r < 0.6: tx, ty = next_wp(ag, rng, rng.uniform(10e3, 35e3), 45); yield from fw_goto(ag, tx, ty, vary(rng, ag.Vc, .08), ag.zc, Tmax=Tm - (ag.t - T0))
                    elif r < 0.8: yield from fw_leg(ag, ag.psi + rng.uniform(-.4, .4), rng.uniform(40, 90), ag.Vc, clip(ag.zc + rng.uniform(-600, 600), 8000, 12000))
                    else: yield from fw_leg(ag, ag.psi, rng.uniform(30, 80), vary(rng, ag.Vc, .1), ag.zc)
                ag.acting = "FIGHTER_AIRCRAFT"; ag.glat = 3.0; ag.a_acc = 6.0; ag.drag_k = 0.07; ag.vz_max = 180; ag.phase = "high_speed_run"
                yield from ramp_speed(ag, rng, rng.uniform(450, 700), style)
                yield from hard_turn(rng.uniform(3, 6), rng.choice([-1, 1]) * rng.uniform(0.5, 1.5), 450)
            while True:
                ag.phase = "high_speed_run"; tx, ty = next_wp(ag, rng, rng.uniform(40e3, 100e3), 50)
                yield from fw_goto(ag, tx, ty, rng.uniform(420, 750), rng.uniform(5000, 12000), Tmax=rng.uniform(60, 160))
                yield from hard_turn(rng.uniform(4, 7), rng.choice([-1, 1]) * rng.uniform(0.6, 2.2), 380)
        elif sub == "bfm":
            while True:
                yield from hard_turn(rng.uniform(4, 8.5), rng.choice([-1, 1]) * rng.uniform(0.8, 3.0), rng.uniform(220, 320))
                r = rng.random()
                if r < 0.3:
                    ag.phase = "zoom_climb"; ag.zc = min(ag.z + rng.uniform(1500, 4500), 14000); ag.Vc = 400; ag.gam_max = rad(65); t = 0.0
                    while ag.zc - ag.z > 150 and t < 25: yield; t += ag.dt
                elif r < 0.55:
                    ag.phase = "dive"; ag.zc = max(ag.z - rng.uniform(1500, 4500), 1500); ag.Vc = 450; t = 0.0
                    while ag.z - ag.zc > 150 and t < 25: yield; t += ag.dt
                ag.phase = "extend"; yield from fw_leg(ag, ag.psi, rng.uniform(4, 15), rng.uniform(250, 380), ag.zc)
        elif sub in ("low_level", "climb_cycles"):
            base = zc
            while True:
                ag.phase = "low_level_cruise"; tx, ty = next_wp(ag, rng, rng.uniform(6e3, 18e3), 55, 18)
                yield from fw_goto(ag, tx, ty, rng.uniform(230, 290), clip(base + rng.uniform(-60, 120), 60, 400))
                if rng.random() < 0.4:
                    yield from hard_turn(rng.uniform(2.5, 4.5), rng.choice([-1, 1]) * rng.uniform(0.3, 0.9), 260); ag.phase = "jink"
                if sub == "climb_cycles" and rng.random() < 0.55:
                    ag.phase = "climb_cycle"; ag.zc = ag.z + rng.uniform(1500, 2800); ag.Vc = 300; ag.gam_max = rad(45); t = 0.0
                    while ag.zc - ag.z > 150 and t < 25: yield; t += ag.dt
                    ag.phase = "descent_cycle"; ag.zc = rng.uniform(300, 600); ag.Vc = 330; t = 0.0
                    while ag.z - ag.zc > 100 and t < 30: yield; t += ag.dt
                    yield from hard_turn(rng.uniform(4, 6), rng.choice([-1, 1]) * rng.uniform(1.5, 3.0), 300); ag.phase = "recovery_turn"; ag.zc = base
        else:
            while True:
                ag.phase = "transit"; tx, ty = next_wp(ag, rng, rng.uniform(20e3, 70e3), 70)
                yield from fw_goto(ag, tx, ty, vary(rng, Vc, .15), clip(zc + rng.uniform(-1500, 1500), 3000, 13000))
    return ag, mission(), dict(subtype=sub + ("_disguised" if disguised else ""), geometry=mode, disguised=disguised, mimic="CIVILIAN_AIRCRAFT" if disguised else "")


# ---------------------------------------------------------------- HELICOPTER
def make_helicopter(rng, dt, p_disguise=0.0):
    sub = rng.choices(["transit", "observation", "search", "shuttle", "approach_land"], [0.3, 0.25, 0.2, 0.1, 0.15])[0]
    cruiseV = rng.uniform(40, 70)
    P = dict(a_max=rng.uniform(2.0, 3.4), tau_v=2.5, tau_a=1.0, vz_max=rng.uniform(4, 8), az_max=2.0, tau_vz=1.5, zmin=0.0)
    ag = Agent("vec", rng, dt, **P); ag.acting = "HELICOPTER"
    z0 = rng.uniform(40, 1200) if sub != "approach_land" else rng.uniform(150, 500)
    rmax = min(60e3, horizon(z0) * 0.9) if sub != "approach_land" else min(20e3, horizon(60) * 0.9)
    psi, mode = place(ag, rng, 3e3, rmax)
    vec_init(ag, psi, cruiseV if sub in ("transit", "approach_land", "search") else 0.0, z0)
    def stop_and_hover(T, amp=None):
        ag.phase = "decel"; ag.vcx = ag.vcy = 0.0
        while math.hypot(ag.vx, ag.vy) > 0.6: yield
        ag.phase = "hover"; ag.zc = ag.z; yield from vec_hover(ag, T, amp)
    def mission():
        if sub == "transit":
            while True:
                ag.phase = "transit"; tx, ty = next_wp(ag, rng, rng.uniform(3e3, 12e3), 60)
                zt = clip(ag.z + rng.uniform(-150, 150), 30, 1500)
                yield from vec_goto(ag, tx, ty, cruiseV, zt, tol=250, stop=False)
                if rng.random() < 0.2: yield from stop_and_hover(rng.uniform(15, 90))
        elif sub == "observation":
            yield from stop_and_hover(rng.uniform(40, 200))
            while True:
                ag.phase = "reposition"; tx, ty = next_wp(ag, rng, rng.uniform(300, 2500), 180, 60)
                yield from vec_goto(ag, tx, ty, rng.uniform(8, 35), clip(ag.z + rng.uniform(-40, 60), 30, 800), tol=15, stop=True)
                yield from stop_and_hover(rng.uniform(30, 240))
        elif sub == "search":
            while True:
                ag.phase = "search_orbit"; R = rng.uniform(300, 1500); c = (ag.x + R * math.cos(ag.hdg + 1.57), ag.y + R * math.sin(ag.hdg + 1.57))
                ccw = rng.random() < 0.5; yield from vec_orbit(ag, c[0], c[1], R, ccw, rng.uniform(22, 38), ag.z, rng.uniform(80, 240))
                ag.phase = "transit"; tx, ty = next_wp(ag, rng, rng.uniform(2e3, 6e3), 60); yield from vec_goto(ag, tx, ty, cruiseV, ag.z, tol=200, stop=False)
                if rng.random() < 0.4: yield from stop_and_hover(rng.uniform(20, 120))
        elif sub == "shuttle":
            while True:
                ag.phase = "hover_taxi"; hx, hy = next_wp(ag, rng, rng.uniform(80, 500), 180, 70)
                yield from vec_goto(ag, hx, hy, rng.uniform(3, 12), ag.z + rng.uniform(-10, 10), tol=3, stop=True)
                yield from stop_and_hover(rng.uniform(10, 50))
        else:
            ag.phase = "approach"; tx, ty = next_wp(ag, rng, rng.uniform(2e3, 5e3), 20, 10)
            yield from vec_goto(ag, tx, ty, cruiseV * 0.7, ag.z * 0.5, tol=10, stop=True)
            yield from stop_and_hover(rng.uniform(5, 25), amp=0.2)
            ag.phase = "vertical_descent"; yield from vec_vertical(ag, 4.0, rng.uniform(1.2, 3.0))
            ag.phase = "landed_hover"; ag.zc = 4.0; yield from vec_hover(ag, 20, amp=0.1, alt_amp=0.1)
            ag.done = True
    return ag, mission(), dict(subtype=sub, geometry=mode)



# ---------------------------------------------------------------- DRONE
def make_drone(rng, dt, p_disguise=0.45):
    disguised_req = rng.random() < p_disguise
    sub = "loitering_uav" if disguised_req else rng.choices(["multirotor", "fixed_wing", "male_hale", "loitering_uav"], [0.38, 0.22, 0.14, 0.26])[0]
    disguised = False; mimic = ""; style = ""
    if sub == "multirotor":
        P = dict(a_max=rng.uniform(3.0, 7.0), tau_v=1.2, tau_a=0.6, vz_max=rng.uniform(3, 6), az_max=3.0, tau_vz=0.8, zmin=0.0)
        ag = Agent("vec", rng, dt, **P); z0 = rng.uniform(20, 450)
        psi, mode = place(ag, rng, 1e3, min(15e3, horizon(z0) * 0.9)); vec_init(ag, psi, 0.0 if rng.random() < 0.4 else rng.uniform(3, 12), z0)
    elif sub == "fixed_wing":
        P = dict(Vmin=16, Vmax=60, a_acc=1.5, a_dec=1.5, glat=rng.uniform(0.35, 0.6), vz_max=rng.uniform(3, 5), az_max=2.0, tau_om=1.0,
                 tau_psi=2.5, tau_a=1.0, tau_v=4.0, tau_g=3.0, tau_pitch=1.2, zmin=10, gam_max=rad(15))
        ag = Agent("fw", rng, dt, **P); set_air_wind(ag, rng, 12); z0 = rng.uniform(100, 2500); Vc = rng.uniform(22, 42)
        psi, mode = place(ag, rng, 3e3, min(40e3, horizon(z0) * 0.9)); fw_init(ag, psi, Vc, z0)
    elif sub == "male_hale":
        P = dict(Vmin=40, Vmax=170, a_acc=0.8, a_dec=1.0, glat=rng.uniform(0.15, 0.3), vz_max=rng.uniform(3, 8), az_max=1.0, tau_om=2.5,
                 tau_psi=6.0, tau_g=4.0, tau_pitch=2.0, zmin=50, gam_max=rad(10))
        ag = Agent("fw", rng, dt, **P); set_air_wind(ag, rng, 25); z0 = rng.uniform(4000, 15000); Vc = rng.uniform(55, 105)
        psi, mode = place(ag, rng, 10e3, min(150e3, horizon(z0) * 0.9)); fw_init(ag, psi, Vc, z0)
    else:
        P = dict(Vmin=28, Vmax=200, a_acc=3.0, a_dec=3.0, glat=rng.uniform(1.5, 3.5), vz_max=30, az_max=3 * G, tau_om=0.8, tau_psi=2.0,
                 tau_a=1.0, tau_v=4.0, tau_g=2.0, tau_pitch=1.0, zmin=3, gam_max=rad(40))
        ag = Agent("fw", rng, dt, **P); set_air_wind(ag, rng, 12); z0 = rng.uniform(100, 1500); Vc = rng.uniform(45, 100)
        psi, mode = place(ag, rng, 3e3, min(40e3, horizon(z0) * 0.9)); fw_init(ag, psi, Vc, z0)
        disguised = disguised_req; mimic = rng.choice(["HELICOPTER", "CIVILIAN_AIRCRAFT"]) if disguised else ""; style = rng.choice(["abrupt", "gradual"]) if disguised else ""
    ag.acting = "DRONE"
    def mission():
        if sub == "multirotor":
            while True:
                r = rng.random()
                if r < 0.22:
                    ag.phase = "decel"; ag.vcx = ag.vcy = 0.0
                    while math.hypot(ag.vx, ag.vy) > 0.4: yield
                    ag.phase = "hover"; ag.zc = ag.z; yield from vec_hover(ag, rng.uniform(10, 70), amp=rng.uniform(0.2, 0.8), alt_amp=rng.uniform(0.1, 0.4))
                elif r < 0.45:
                    ag.phase = "waypoint_stop_go"; hx, hy = next_wp(ag, rng, rng.uniform(50, 500), 180, 60)
                    yield from vec_goto(ag, hx, hy, rng.uniform(4, 15), clip(ag.z + rng.uniform(-30, 40), 15, 500), tol=3, stop=True)
                    ag.phase = "hover"; ag.zc = ag.z; yield from vec_hover(ag, rng.uniform(5, 30), amp=rng.uniform(0.2, 0.7))
                elif r < 0.62:
                    ag.phase = "survey"; sp = rng.uniform(4, 8); Lleg = rng.uniform(150, 500); gap = rng.uniform(25, 60); hd = rng.uniform(0, 6.28); ex, ey = math.cos(hd), math.sin(hd)
                    for k in range(rng.randint(2, 4)):
                        s = 1 if k % 2 == 0 else -1
                        yield from vec_goto(ag, ag.x + s * ex * Lleg, ag.y + s * ey * Lleg, sp, ag.z, tol=12, stop=False)
                        yield from vec_goto(ag, ag.x - ey * gap, ag.y + ex * gap, sp * 0.7, ag.z, tol=8, stop=False)
                elif r < 0.78:
                    ag.phase = "orbit_poi"; R = rng.uniform(30, 150); sp = rng.uniform(3, 8); hd = ag.hdg
                    c = (ag.x + R * math.cos(hd + 1.57), ag.y + R * math.sin(hd + 1.57)); yield from vec_orbit(ag, c[0], c[1], R, rng.random() < .5, sp, ag.z, rng.uniform(40, 120))
                elif r < 0.9:
                    ag.phase = "vertical"; yield from vec_vertical(ag, clip(ag.z + rng.choice([-1, 1]) * rng.uniform(30, 150), 15, 500), rng.uniform(2, 5))
                elif r < 0.96:
                    ag.phase = "transit"; hx, hy = next_wp(ag, rng, rng.uniform(300, 1200), 90, 40)
                    yield from vec_goto(ag, hx, hy, rng.uniform(8, 18), clip(ag.z + rng.uniform(-40, 40), 15, 500), tol=25, stop=False)
                else:
                    ag.phase = "fast_dash"; ag.a_max = 9.0; hx, hy = next_wp(ag, rng, rng.uniform(300, 900), 40, 20)
                    yield from vec_goto(ag, hx, hy, rng.uniform(20, 32), ag.z, tol=30, stop=False); ag.a_max = P["a_max"]
        elif sub == "fixed_wing":
            while True:
                r = rng.random(); Vn = vary(rng, Vc, .15)
                if r < 0.3:
                    ag.phase = "loiter"; R = rng.uniform(150, 500); c = (ag.x + R * math.cos(ag.psi + 1.57), ag.y + R * math.sin(ag.psi + 1.57))
                    yield from fw_orbit(ag, c[0], c[1], R, rng.random() < .5, Vn, ag.z, rng.uniform(60, 220))
                elif r < 0.62:
                    ag.phase = "waypoint"; tx, ty = next_wp(ag, rng, rng.uniform(1e3, 5e3), 120, 50)
                    yield from fw_goto(ag, tx, ty, Vn, clip(ag.z + rng.uniform(-200, 250), 60, 3000))
                elif r < 0.8:
                    ag.phase = "survey"; Lleg = rng.uniform(1.5e3, 3.5e3); gap = rng.uniform(250, 500); ex, ey = math.cos(ag.psi), math.sin(ag.psi); sg = 1
                    for k in range(rng.randint(2, 4)):
                        yield from fw_goto(ag, ag.x + ex * Lleg * sg, ag.y + ey * Lleg * sg, Vn, ag.z)
                        yield from fw_goto(ag, ag.x - ey * gap, ag.y + ex * gap, Vn, ag.z, tol=150); sg = -sg
                else:
                    ag.phase = "climb" if rng.random() < .5 else "descent"; yield from fw_alt(ag, clip(ag.z + (1 if ag.phase == "climb" else -1) * rng.uniform(150, 600), 60, 3000), Vn, tol=40, T=120)
        elif sub == "male_hale":
            while True:
                r = rng.random()
                if r < 0.35: ag.phase = "racetrack"; yield from fw_racetrack(ag, ag.psi, rng.uniform(10e3, 30e3), vary(rng, Vc, .1), ag.z, rng.uniform(300, 700), right=rng.random() < .5)
                elif r < 0.6:
                    ag.phase = "orbit"; R = rng.uniform(3e3, 8e3); c = (ag.x + R * math.cos(ag.psi + 1.57), ag.y + R * math.sin(ag.psi + 1.57))
                    yield from fw_orbit(ag, c[0], c[1], R, rng.random() < .5, Vc, ag.z, rng.uniform(300, 700))
                elif r < 0.85:
                    ag.phase = "transit"; tx, ty = next_wp(ag, rng, rng.uniform(15e3, 50e3), 60); yield from fw_goto(ag, tx, ty, vary(rng, Vc, .15), ag.z)
                else:
                    ag.phase = "climb" if rng.random() < .5 else "descent"; yield from fw_alt(ag, clip(ag.z + (1 if ag.phase == "climb" else -1) * rng.uniform(500, 2500), 3000, 15000), Vc, tol=100, T=400)
        else:
            if disguised and mimic == "HELICOPTER":
                ag.acting = "HELICOPTER"; ag.phase = "mimic_helicopter"; vm = rng.uniform(38, 55); ag.glat = 0.35
                ag.Vc = ag.V = vm; ag.zc = ag.z = rng.uniform(100, 600); T0 = ag.t; Tm = rng.uniform(150, 400)
                while ag.t - T0 < Tm:
                    if rng.random() < .5: tx, ty = next_wp(ag, rng, rng.uniform(1.5e3, 4e3), 60); yield from fw_goto(ag, tx, ty, vary(rng, vm, .15), ag.z, Tmax=Tm - (ag.t - T0))
                    else:
                        c = (ag.x + 400 * math.cos(ag.psi + 1.57), ag.y + 400 * math.sin(ag.psi + 1.57)); yield from fw_orbit(ag, c[0], c[1], 400, rng.random() < .5, vm, ag.z, min(120, max(Tm - (ag.t - T0), 5)))
                ag.acting = "DRONE"; ag.glat = P["glat"]; ag.phase = "breakout_dash"; yield from ramp_speed(ag, rng, rng.uniform(95, 150), style)
            elif disguised and mimic == "CIVILIAN_AIRCRAFT":
                ag.acting = "CIVILIAN_AIRCRAFT"; ag.phase = "mimic_civilian"; vm = rng.uniform(50, 75); ag.glat = 0.3
                ag.Vc = ag.V = vm; ag.zc = ag.z = rng.uniform(500, 3500); T0 = ag.t; Tm = rng.uniform(150, 420)
                while ag.t - T0 < Tm:
                    tx, ty = next_wp(ag, rng, rng.uniform(4e3, 14e3), 50); yield from fw_goto(ag, tx, ty, vary(rng, vm, .1), clip(ag.z + rng.uniform(-300, 300), 300, 3800), Tmax=Tm - (ag.t - T0))
                ag.acting = "DRONE"; ag.glat = P["glat"]; ag.phase = "breakout_dash"; yield from ramp_speed(ag, rng, rng.uniform(100, 150), style)
            for _ in range(rng.randint(1, 3)):
                ag.phase = "transit"; tx, ty = next_wp(ag, rng, rng.uniform(2e3, 6e3), 80, 30); yield from fw_goto(ag, tx, ty, rng.uniform(55, 120), clip(ag.z + rng.uniform(-300, 300), 100, 1800))
            while True:
                r = rng.random()
                if r < 0.35:
                    ag.phase = "loiter"; R = rng.uniform(400, 1000); c = (ag.x + R * math.cos(ag.psi + 1.57), ag.y + R * math.sin(ag.psi + 1.57))
                    yield from fw_orbit(ag, c[0], c[1], R, rng.random() < .5, rng.uniform(45, 85), ag.z, rng.uniform(40, 140))
                elif r < 0.65:
                    ag.phase = "transit"; tx, ty = next_wp(ag, rng, rng.uniform(1.5e3, 5e3), 90, 35); yield from fw_goto(ag, tx, ty, rng.uniform(55, 120), clip(ag.z + rng.uniform(-300, 300), 100, 1800))
                elif r < 0.85:
                    ag.phase = "descent" if rng.random() < .5 else "climb"; yield from fw_alt(ag, clip(ag.z + (-1 if ag.phase == "descent" else 1) * rng.uniform(100, 500), 80, 1800), vary(rng, 80, .3), tol=30, T=100)
                else:
                    ag.phase = "dash"; ag.a_acc = 4.0; yield from fw_leg(ag, ag.psi + rng.uniform(-.5, .5), rng.uniform(15, 40), rng.uniform(110, 170), ag.z); ag.a_acc = 3.0
    return ag, mission(), dict(subtype=sub + ("_disguised" if disguised else ""), geometry=mode, disguised=disguised, mimic=mimic)


# ---------------------------------------------------------------- MISSILE CLASSES: generic powered-flight profiles
# Scripted flight shapes only: launch / boost / sustain / coast / cruise legs with course, speed and altitude changes.
# There is NO guidance, homing, target, evasion or terminal-attack logic here.
def make_surface_missile(rng, dt, p_disguise=0.0):
    sub = rng.choices(["sea_skimming_cruise", "supersonic_cruise", "terrain_following_cruise", "launch_climb_cruise", "high_altitude_cruise"], [0.27, 0.17, 0.22, 0.19, 0.15])[0]
    start_z = None
    if sub == "sea_skimming_cruise": Vc, zc, Vmax, zmin = rng.uniform(220, 300), rng.uniform(8, 25), 340, 3
    elif sub == "supersonic_cruise": Vc, zc, Vmax, zmin = rng.uniform(600, 950), rng.uniform(10, 120), 1100, 5
    elif sub == "terrain_following_cruise": Vc, zc, Vmax, zmin = rng.uniform(200, 270), rng.uniform(70, 200), 320, 15
    elif sub == "launch_climb_cruise": Vc, zc, Vmax, zmin = rng.uniform(220, 300), rng.uniform(1500, 5000), 340, 15; start_z = rng.uniform(20, 60)
    else: Vc, zc, Vmax, zmin = rng.uniform(230, 300), rng.uniform(3000, 9000), 340, 15
    P = dict(Vmin=30, Vmax=Vmax, a_acc=10.0, a_dec=5.0, glat=rng.uniform(3, 6), vz_max=60, az_max=4 * G, tau_om=0.6, tau_psi=2.0, tau_a=0.8,
             tau_v=5.0, tau_g=1.5, tau_pitch=0.8, zmin=zmin, gam_max=rad(25))
    ag = Agent("fw", rng, dt, **P); ag.acting = "SURFACE_MISSILE"; set_air_wind(ag, rng, 8)
    boosted = sub == "launch_climb_cruise" or rng.random() < 0.3
    z_init = start_z if start_z is not None else (zc * 2 + 20 if boosted else zc)
    psi, mode = place(ag, rng, 3e3, min(60e3, horizon(max(z_init, zc)) * 0.9)); fw_init(ag, psi, 40.0 if sub == "launch_climb_cruise" else (90.0 if boosted else Vc), z_init)
    zmax = {"sea_skimming_cruise": 40, "supersonic_cruise": 250, "terrain_following_cruise": 300, "launch_climb_cruise": 6000, "high_altitude_cruise": 10000}[sub]
    zmin_c = {"sea_skimming_cruise": 6, "supersonic_cruise": 10, "terrain_following_cruise": 50, "launch_climb_cruise": 1200, "high_altitude_cruise": 3000}[sub]
    def mission():
        if boosted:
            ag.phase = "boost"; ag.al_cmd = rng.uniform(25, 45); ag.zc = zc
            if sub == "launch_climb_cruise": ag.gam_max = rad(60); ag.vz_max = 110; ag.az_max = 6 * G
            while ag.V < 0.85 * Vc and ag.t < 14: yield
            ag.al_cmd = None
            if sub == "launch_climb_cruise":
                ag.phase = "launch_climb"; t = 0.0
                while abs(ag.z - zc) > 200 and t < 90: yield; t += ag.dt
                ag.gam_max = rad(25); ag.vz_max = 60; ag.az_max = 4 * G
        t_tf = 0.0; lam = rng.uniform(25, 60); A = rng.uniform(15, 45)
        while True:
            r = rng.random(); Vn = clip(vary(rng, Vc, 0.1), 120, Vmax * 0.97)
            if sub == "terrain_following_cruise" and r < 0.5:
                ag.phase = "terrain_following"; T = rng.uniform(20, 50); n = int(T / ag.dt); psi_t = ag.psi + rng.uniform(-.2, .2)
                for i in range(n): ag.psic = psi_t; ag.Vc = Vn; ag.zc = clip(zc + A * math.sin(2 * math.pi * (t_tf + i * ag.dt) / lam), zmin_c, zmax); yield
                t_tf += T
            elif r < 0.45:
                ag.phase = "cruise"; yield from fw_leg(ag, ag.psi + rng.uniform(-0.25, 0.25), rng.uniform(15, 45), Vn, clip(ag.z, zmin_c, zmax))
            elif r < 0.72:
                ag.phase = "midcourse_turn"; ag.glat = min(P["glat"] * 1.3, 8.0); yield from fw_leg(ag, ag.psi + rng.choice([-1, 1]) * rng.uniform(0.4, 1.4), rng.uniform(15, 35), Vn, clip(ag.z, zmin_c, zmax)); ag.glat = P["glat"]
            elif r < 0.87:
                ag.phase = "speed_change"; yield from fw_leg(ag, ag.psi, rng.uniform(20, 40), clip(vary(rng, Vc, 0.22), 120, Vmax * 0.97), clip(ag.z, zmin_c, zmax))
            else:
                z2 = clip(ag.z * rng.uniform(0.55, 1.7) + rng.uniform(-5, 15), zmin_c, zmax); ag.phase = "climb" if z2 > ag.z else "descent"
                yield from fw_leg(ag, ag.psi + rng.uniform(-.15, .15), rng.uniform(20, 40), Vn, z2)
    return ag, mission(), dict(subtype=sub, geometry=mode)

def make_aerial_missile(rng, dt, p_disguise=0.0):
    sub = rng.choices(["short_range_boost_coast", "medium_range_boost_sustain_coast", "vertical_launch_pitchover", "long_range_cruise"], [0.3, 0.3, 0.2, 0.2])[0]
    if sub == "vertical_launch_pitchover": z0, V0, g0 = rng.uniform(20, 200), 40.0, rad(80)
    elif sub == "long_range_cruise": z0, V0, g0 = rng.uniform(8000, 14000), rng.uniform(250, 320), 0.0
    else: z0, V0, g0 = rng.uniform(3000, 10000), rng.uniform(200, 300), 0.0
    P = dict(Vmin=100 if sub != "vertical_launch_pitchover" else 20, Vmax=1500, glat=rng.uniform(4, 9), tau_om=0.4, tau_psi=2.0, tau_a=0.5, az_max=5 * G, tau_g=1.5,
             tau_pitch=0.6, zmin=100, gam_max=rad(80) if sub == "vertical_launch_pitchover" else rad(30), vz_max=400 if sub == "vertical_launch_pitchover" else 200)
    ag = Agent("fw", rng, dt, **P); ag.acting = "AERIAL_MISSILE"; set_air_wind(ag, rng, 5)
    psi, mode = place(ag, rng, 5e3, min(90e3, horizon(z0 + 500) * 0.9)); fw_init(ag, psi, V0, z0); ag.gam = g0
    ab = rng.uniform(12, 22) * G; k = 1.2 * G / 700.0 ** 2
    tb = {"short_range_boost_coast": rng.uniform(2.5, 5), "medium_range_boost_sustain_coast": rng.uniform(3, 5.5), "vertical_launch_pitchover": rng.uniform(5, 8), "long_range_cruise": rng.uniform(4, 7)}[sub]
    ts = {"short_range_boost_coast": 0.0, "medium_range_boost_sustain_coast": rng.uniform(8, 25), "vertical_launch_pitchover": rng.uniform(0, 10), "long_range_cruise": rng.uniform(40, 100)}[sub]
    a_s = rng.uniform(0.5, 2.5) * G; V_s = rng.uniform(550, 900)
    zt0 = rng.uniform(3000, 9000) if sub == "vertical_launch_pitchover" else z0
    def drag(): return k * math.exp(-ag.z / 8500.0) * ag.V ** 2
    def mission():
        t0 = ag.t; psi0 = ag.psi; nxt = 0.0; ag.zc = zt0
        while True:
            te = ag.t - t0
            if te < tb: ag.phase = "boost"; ag.al_cmd = ab - drag()
            elif te < tb + ts:
                ag.phase = "sustain"
                if sub == "long_range_cruise": ag.al_cmd = clip((V_s - ag.V) / 5.0, -drag(), 6.0)
                else: ag.al_cmd = a_s - drag()
            else:
                ag.phase = "coast"; ag.al_cmd = -drag()
                if ag.V < 160: ag.done = True; return
            if te > 3.0 and te >= nxt:
                nxt = te + rng.uniform(3, 8); ag.psic = psi0 + rng.uniform(-0.6, 0.6) if sub != "vertical_launch_pitchover" or te > 10 else psi0
                if rng.random() < 0.35: ag.zc = clip(ag.z + rng.uniform(-700, 900), 600, 14000)
                if sub == "vertical_launch_pitchover" and te > 8: ag.zc = clip(ag.z + rng.uniform(-300, 300), 2500, 12000)
            yield
    return ag, mission(), dict(subtype=sub, geometry=mode)

# ---------------------------------------------------------------- SURFACE VESSELS (skiff / boat / ship) with disguises
REPERTOIRE = {"SKIFF": (["transit", "zigzag", "fishing", "dash", "circle", "stop_go"], [0.22, 0.16, 0.18, 0.14, 0.15, 0.15]),
              "BOAT": (["transit", "patrol", "fishing", "dash", "circle", "stop_go", "zigzag"], [0.2, 0.18, 0.16, 0.1, 0.14, 0.12, 0.1]),
              "SHIP": (["transit", "zigzag", "drift", "racetrack", "anchored", "slow_down_speed_up"], [0.3, 0.17, 0.12, 0.13, 0.08, 0.2])}
def make_surface(rng, dt, cls, p_disguise=0.45):
    if cls == "SKIFF":
        mimic_cls = "BOAT"; sub = rng.choices(["transit", "zigzag", "fishing", "dash", "circle", "stop_go"], [0.25, 0.15, 0.2, 0.12, 0.13, 0.15])[0]
        P = dict(Vmin=0.0, Vmax=32, a_acc=rng.uniform(1.0, 3.0), a_dec=rng.uniform(1.5, 3.0), om_cap=rng.uniform(0.2, 0.5), tau_v=3.0, tau_a=1.5,
                 tau_om=1.2, tau_psi=2.5, zmin=0.5); z_ref = rng.uniform(1, 3); hv = [(rng.uniform(0.3, 0.8), rng.uniform(3, 6), rng.uniform(0, 6.28)), (rng.uniform(0.1, 0.3), rng.uniform(2, 4), rng.uniform(0, 6.28))]
        Vtr = (8, 20); Vlow = (0.3, 2.5)
    elif cls == "BOAT":
        mimic_cls = rng.choice(["SHIP", "SKIFF"]); sub = rng.choices(["transit", "patrol", "fishing", "dash", "circle", "stop_go"], [0.25, 0.2, 0.2, 0.1, 0.13, 0.12])[0]
        P = dict(Vmin=0.0, Vmax=40, a_acc=rng.uniform(0.4, 1.0), a_dec=rng.uniform(0.5, 1.2), om_cap=rng.uniform(0.06, 0.15), tau_v=6.0, tau_a=3.0,
                 tau_om=3.0, tau_psi=6.0, zmin=0.5); z_ref = rng.uniform(3, 8); hv = [(rng.uniform(0.2, 0.6), rng.uniform(4, 7), rng.uniform(0, 6.28)), (rng.uniform(0.1, 0.3), rng.uniform(3, 5), rng.uniform(0, 6.28))]
        Vtr = (5, 18); Vlow = (0.5, 3.0)
    else:
        mimic_cls = None; sub = rng.choices(["transit", "zigzag", "drift", "racetrack", "anchored", "slow_down_speed_up"], [0.35, 0.15, 0.13, 0.15, 0.1, 0.12])[0]
        P = dict(Vmin=0.0, Vmax=22, a_acc=rng.uniform(0.04, 0.15), a_dec=rng.uniform(0.05, 0.2), om_cap=rng.uniform(0.006, 0.03), tau_v=60.0, tau_a=20.0,
                 tau_om=20.0, tau_psi=60.0, zmin=0.5); z_ref = rng.uniform(8, 35); hv = [(rng.uniform(0.05, 0.3), rng.uniform(7, 12), rng.uniform(0, 6.28))]
        Vtr = (4, 15); Vlow = (0.1, 1.0)
    disguised = bool(mimic_cls) and rng.random() < p_disguise; style = rng.choice(["abrupt", "gradual"]) if disguised else ""
    ag = Agent("fw", rng, dt, planar=True, **P); ag.acting = cls; ag.z_ref = z_ref; ag.heave = hv; ag.zc = z_ref
    cur = rng.uniform(0, 0.6); d = rng.uniform(0, 6.28); ag.wind = (cur * math.cos(d), cur * math.sin(d)); ag.wind_ref = 4.0
    psi, mode = place(ag, rng, 3e3, min(60e3, horizon(z_ref) * 0.9)); V0 = rng.uniform(*Vtr)
    fw_init(ag, psi, V0, z_ref); ag.z = z_ref; real_om = ag.om_cap
    def mission():
        nonlocal sub
        if disguised:
            ag.acting = mimic_cls; ag.phase = "mimic_" + mimic_cls.lower()
            if mimic_cls == "SKIFF":
                ag.om_cap = 0.3; Vm = rng.uniform(2, 8); ag.V = ag.Vc = Vm; T = rng.uniform(250, 600)
                while ag.t < T: yield from fw_leg(ag, ag.psi + rng.uniform(-1.2, 1.2), rng.uniform(15, 60), vary(rng, Vm, .4), z_ref)
            else:
                ag.om_cap = 0.015 if mimic_cls == "SHIP" else 0.1; Vm = rng.uniform(1.5, 3.0) if cls == "SKIFF" else rng.uniform(5, 9)
                ag.V = ag.Vc = Vm; ag.a_acc = min(ag.a_acc, 0.3); T = rng.uniform(250, 600)
                while ag.t < T:
                    tx, ty = next_wp(ag, rng, rng.uniform(1.5e3, 5e3), 40, 18); yield from fw_goto(ag, tx, ty, vary(rng, Vm, .2), z_ref, tol=100, Tmax=T - ag.t)
            ag.acting = cls; ag.om_cap = real_om; ag.a_acc = P["a_acc"]; ag.phase = "breakout_dash"
            vt = rng.uniform(20, 30) if cls == "SKIFF" else rng.uniform(18, 34) if mimic_cls == "SHIP" else rng.uniform(10, 16)
            yield from ramp_speed(ag, rng, vt, style); yield from fw_leg(ag, ag.psi + rng.uniform(-.4, .4), rng.uniform(40, 120), vt, z_ref)
        names, wts = REPERTOIRE[cls]
        while True:
            if rng.random() < 0.75: sub = rng.choices(names, wts)[0]
            ag.phase = sub
            if sub in ("transit", "patrol"):
                tx, ty = next_wp(ag, rng, rng.uniform(700, 3500) if cls != "SHIP" else rng.uniform(2e3, 9e3), 80, 28)
                yield from fw_goto(ag, tx, ty, rng.uniform(*Vtr), z_ref, tol=150)
            elif sub == "zigzag":
                base = ag.psi
                for _ in range(rng.randint(3, 8)):
                    yield from fw_leg(ag, base + rng.choice([-1, 1]) * rng.uniform(0.3, 1.0 if cls == "SKIFF" else 0.45), rng.uniform(10, 40) if cls == "SKIFF" else rng.uniform(150, 400), rng.uniform(*Vtr), z_ref)
            elif sub in ("fishing", "drift", "anchored"):
                v = rng.uniform(*Vlow) if sub == "fishing" else rng.uniform(0.05, 0.6)
                for _ in range(rng.randint(2, 5)): yield from fw_leg(ag, ag.psi + rng.uniform(-1.2, 1.2), rng.uniform(40, 200) if cls != "SHIP" else rng.uniform(200, 600), v, z_ref)
            elif sub == "stop_go":
                yield from fw_leg(ag, ag.psi, rng.uniform(20, 60), rng.uniform(*Vtr), z_ref); yield from fw_leg(ag, ag.psi + rng.uniform(-.6, .6), rng.uniform(15, 50), rng.uniform(0.2, 1.0), z_ref)
            elif sub == "slow_down_speed_up":
                yield from fw_leg(ag, ag.psi, rng.uniform(150, 400), rng.uniform(1.0, 3.0), z_ref); yield from fw_leg(ag, ag.psi + rng.uniform(-.2, .2), rng.uniform(150, 400), rng.uniform(*Vtr), z_ref)
            elif sub == "dash":
                tx, ty = next_wp(ag, rng, rng.uniform(3e3, 8e3), 60, 20); yield from fw_goto(ag, tx, ty, rng.uniform(22, 31) if cls == "SKIFF" else rng.uniform(20, 36), z_ref, tol=150, Tmax=200)
                yield from fw_leg(ag, ag.psi, rng.uniform(20, 60), rng.uniform(*Vlow), z_ref)
            elif sub == "circle":
                R = rng.uniform(80, 300) if cls == "SKIFF" else rng.uniform(200, 700); c = (ag.x + R * math.cos(ag.psi + 1.57), ag.y + R * math.sin(ag.psi + 1.57))
                yield from fw_orbit(ag, c[0], c[1], R, rng.random() < .5, rng.uniform(4, 12), z_ref, rng.uniform(60, 200))
            elif sub == "racetrack":
                yield from fw_racetrack(ag, ag.psi, rng.uniform(5e3, 10e3), rng.uniform(8, 14), z_ref, rng.uniform(500, 1200), right=rng.random() < .5, tol=200)
    return ag, mission(), dict(subtype=sub + ("_disguised" if disguised else ""), geometry=mode, disguised=disguised, mimic=mimic_cls if disguised else "")

In [ ]:
# ---------------- Generator part 4: labels, motion states, nominal uncertainty ----------------
# =========================================================================================
#  behaviour vocabulary (ground-truth annotation only; never a model input)
# =========================================================================================
PHASE_INFO = {
 # name: (group, plain-language meaning)
 "transit": ("Cruise / transit", "Travelling from one point to the next along fairly straight legs with gentle turns at waypoints."),
 "cruise": ("Cruise / transit", "Steady flight on a straight leg at a constant speed and height."),
 "waypoint": ("Cruise / transit", "Fixed-wing drone flying to its next programmed point."),
 "patrol": ("Cruise / transit", "Boat moving between patrol points, steady speed, moderate turns."),
 "low_level_cruise": ("Cruise / transit", "Flying low and fast along a leg, with small height changes."),
 "extend": ("Cruise / transit", "Short straight, level run after a hard manoeuvre to regain speed."),
 "holding": ("Orbit / holding", "Airliner waiting in a race-track oval (straight leg, 180 deg turn, straight leg, 180 deg turn)."),
 "orbit": ("Orbit / holding", "Flying a full circle around a point at constant speed."),
 "racetrack": ("Orbit / holding", "Oval pattern: two straight legs joined by two half-circles."),
 "cap_racetrack": ("Orbit / holding", "Fighter combat-air-patrol: a long race-track at height and moderate speed."),
 "loiter": ("Orbit / holding", "Drone circling slowly over an area."),
 "orbit_poi": ("Orbit / holding", "Multirotor circling a point of interest at small radius."),
 "search_orbit": ("Orbit / holding", "Helicopter circling an area while searching."),
 "circle": ("Orbit / holding", "Boat or skiff going round in a circle (for example around a buoy or net)."),
 "hover": ("Hover / stationary", "Staying in one place in the air with tiny drift (rotorcraft, multirotor)."),
 "landed_hover": ("Hover / stationary", "Helicopter holding a very low hover just before touch-down."),
 "decel": ("Slow / stop-and-go", "Slowing down to a stop, usually just before a hover."),
 "hover_taxi": ("Slow / stop-and-go", "Helicopter sliding a short distance at low speed close to the ground, then stopping."),
 "reposition": ("Slow / stop-and-go", "Helicopter moving to a new observation spot, then stopping."),
 "waypoint_stop_go": ("Slow / stop-and-go", "Multirotor flying to a point, stopping, hovering, then moving again."),
 "stop_go": ("Slow / stop-and-go", "Vessel running, then nearly stopping, then moving again."),
 "slow_down_speed_up": ("Slow / stop-and-go", "Ship cutting speed for a while, then returning to normal speed."),
 "fishing": ("Slow / drift", "Slow trawling with frequent small course changes."),
 "drift": ("Slow / drift", "Almost no propulsion: moving with the current, small heading changes."),
 "anchored": ("Slow / drift", "Almost stationary, swinging slightly around the anchor."),
 "climb": ("Climb / descent", "Gaining height at a steady climb rate."),
 "descent": ("Climb / descent", "Losing height at a steady descent rate."),
 "approach": ("Climb / descent", "Descending and slowing for landing."),
 "final": ("Climb / descent", "Last straight, low, slow stretch before landing."),
 "vertical": ("Climb / descent", "Multirotor going straight up or down."),
 "vertical_descent": ("Climb / descent", "Helicopter sinking straight down to land."),
 "zoom_climb": ("Climb / descent", "Fighter trading speed for height in a steep climb."),
 "dive": ("Climb / descent", "Fighter descending steeply, gaining speed."),
 "climb_cycle": ("Climb / descent", "Fighter climbing from low level to a higher altitude."),
 "descent_cycle": ("Climb / descent", "Fighter coming back down to low level."),
 "launch_climb": ("Climb / descent", "Powered climb after launch up to cruise altitude."),
 "hard_turn": ("Hard manoeuvre", "A high-g turn (several g for fighters)."),
 "jink": ("Hard manoeuvre", "Short sharp direction change."),
 "recovery_turn": ("Hard manoeuvre", "Turn that leaves a climb or descent cycle."),
 "weather_deviation": ("Hard manoeuvre", "Airliner turning off its route to avoid weather, then returning."),
 "midcourse_turn": ("Hard manoeuvre", "A planned course change in the middle of a long flight."),
 "zigzag": ("Hard manoeuvre", "Alternating left/right course changes, regular or evasive."),
 "sprint": ("Speed change / burst", "A short burst of high speed."),
 "high_speed_run": ("Speed change / burst", "Fighter flying fast and straight between turns."),
 "dash": ("Speed change / burst", "A fast burst, then slowing again."),
 "fast_dash": ("Speed change / burst", "Multirotor flying at its top speed for a short distance."),
 "breakout_dash": ("Speed change / burst", "The moment a disguised target drops the disguise and accelerates."),
 "speed_change": ("Speed change / burst", "Straight flight while speeding up or slowing down."),
 "survey": ("Search / survey", "Back-and-forth lawnmower pattern to cover an area."),
 "boost": ("Powered flight", "Rocket motor burning: very high acceleration."),
 "sustain": ("Powered flight", "Lower thrust that roughly holds or slowly changes speed."),
 "coast": ("Powered flight", "No thrust: speed decays because of drag."),
 "terrain_following": ("Powered flight", "Low flight with a gently rolling height profile."),
 "mimic_civilian": ("Disguise (mimic)", "Behaving like an airliner: steady speed and height, gentle turns."),
 "mimic_helicopter": ("Disguise (mimic)", "Behaving like a helicopter: slow, low, loitering."),
 "mimic_boat": ("Disguise (mimic)", "Behaving like a boat: moderate steady speed."),
 "mimic_ship": ("Disguise (mimic)", "Behaving like a ship: slow, steady, very wide turns."),
 "mimic_skiff": ("Disguise (mimic)", "Behaving like a skiff: small, twitchy, frequent turns."),
}
PHASE_GROUP = {k: v[0] for k, v in PHASE_INFO.items()}

def add_motion_state(df):
    """Per-packet motion label computed from the kinematics only (what a radar could see)."""
    sp = df["vel_mag"].clip(lower=1e-6)
    a_al = (df.vel_x * df.accel_x + df.vel_y * df.accel_y + df.vel_z * df.accel_z) / sp
    om = np.degrees((df.vel_x * df.accel_y - df.vel_y * df.accel_x).abs() / (df.vel_x ** 2 + df.vel_y ** 2 + 1.0))
    st = np.full(len(df), "steady cruise", dtype=object)
    st[(a_al.abs() > np.maximum(0.05, 0.002 * sp)).to_numpy()] = "speed change"
    st[(om > 0.1).to_numpy()] = "turning"
    st[(df.vel_z < -np.maximum(1.0, 0.02 * sp)).to_numpy()] = "descending"
    st[(df.vel_z > np.maximum(1.0, 0.02 * sp)).to_numpy()] = "climbing"
    st[(df.vel_mag < 1.5).to_numpy()] = "hover / near-stationary"
    df["motion"] = st; df["turn_deg_s"] = om; df["a_along"] = a_al
    return df

In [ ]:
# ---------------- Generator part 5: track simulation and dataset builder ----------------
# =========================================================================================
#  nominal tracker uncertainty (reference values, NOT measured on a real radar)
# =========================================================================================
Q_JERK = {"CIVILIAN_AIRCRAFT": 1e-4, "FIGHTER_AIRCRAFT": 2e-2, "HELICOPTER": 2e-3, "DRONE": 4e-3, "SURFACE_MISSILE": 2e-2,
          "AERIAL_MISSILE": 5e-1, "SKIFF": 2e-3, "BOAT": 5e-4, "SHIP": 1e-6}
RADAR_SIGMA = dict(range_m=25.0, az_rad=1.0e-3, el_rad=1.0e-3)

def nominal_uncertainty(P, cls, dt=DT_OUT):
    """Covariance-only Kalman recursion (constant-acceleration model, one filter per axis).
    No random numbers: it gives the uncertainty a tracker with these nominal radar accuracies would report along this geometry."""
    n = len(P); r = np.linalg.norm(P, axis=1); th = np.arctan2(P[:, 1], P[:, 0])
    sr, saz, sel = RADAR_SIGMA["range_m"], RADAR_SIGMA["az_rad"], RADAR_SIGMA["el_rad"]
    Rm = np.zeros((n, 3))
    Rm[:, 0] = (sr * np.cos(th)) ** 2 + (r * saz * np.sin(th)) ** 2
    Rm[:, 1] = (sr * np.sin(th)) ** 2 + (r * saz * np.cos(th)) ** 2
    Rm[:, 2] = np.maximum(15.0, r * sel) ** 2
    F = np.array([[1, dt, dt * dt / 2], [0, 1, dt], [0, 0, 1]]); q = Q_JERK[cls]
    Q = q * np.array([[dt ** 5 / 20, dt ** 4 / 8, dt ** 3 / 6], [dt ** 4 / 8, dt ** 3 / 3, dt ** 2 / 2], [dt ** 3 / 6, dt ** 2 / 2, dt]])
    out = np.zeros((n, 3, 3, 3))
    for ax in range(3):
        Pm = np.diag([Rm[0, ax], 80.0 ** 2, 8.0 ** 2])
        for k in range(n):
            if k > 0: Pm = F @ Pm @ F.T + Q
            s = Pm[0, 0] + Rm[k, ax]; K = Pm[:, 0] / s
            Pm = Pm - np.outer(K, Pm[0, :]); out[k, ax] = Pm
    return out

FACTORIES = {"CIVILIAN_AIRCRAFT": make_civilian, "FIGHTER_AIRCRAFT": make_fighter, "HELICOPTER": make_helicopter, "DRONE": make_drone,
             "SURFACE_MISSILE": make_surface_missile, "AERIAL_MISSILE": make_aerial_missile,
             "SKIFF": lambda r, d, p: make_surface(r, d, "SKIFF", p), "BOAT": lambda r, d, p: make_surface(r, d, "BOAT", p), "SHIP": lambda r, d, p: make_surface(r, d, "SHIP", p)}
FINE_DT = {"FIGHTER_AIRCRAFT": 0.125, "SURFACE_MISSILE": 0.1, "AERIAL_MISSILE": 0.05, "HELICOPTER": 0.1, "DRONE": 0.1}
T_RANGE = {"CIVILIAN_AIRCRAFT": (300, 900), "FIGHTER_AIRCRAFT": (200, 500), "HELICOPTER": (200, 600), "DRONE": (200, 600),
           "SURFACE_MISSILE": (150, 450), "AERIAL_MISSILE": (30, 110), "SKIFF": (200, 600), "BOAT": (300, 800), "SHIP": (400, 900)}
MAX_RANGE = 250e3
TRUTH_COLS = ["time_s", "true_class", "acting_class", "phase", "behaviour_group", "subtype", "geometry", "disguised", "disguise_type", "breakout_time_s", "speed", "lateral_g"]

def simulate_track(cls, seed, track_no, t0=1.76e9, min_packets=8, p_disguise=0.45, uncertainty=True):
    """One noise-free track sampled every 2.5 s. Returns (DataFrame, meta)."""
    for attempt in range(30):
        rng = random.Random(seed * 100 + attempt); dt = FINE_DT.get(cls, 0.25)
        ag, gen, meta = FACTORIES[cls](rng, dt, p_disguise)
        Tcap = rng.uniform(*T_RANGE[cls])
        if meta.get("disguised"): Tcap = max(Tcap, 1000.0 if cls in ("SKIFF", "BOAT") else 800.0)
        n_sub = int(round(DT_OUT / dt)); rows = []; k = 0
        while ag.t < Tcap - 1e-9 and not ag.done:
            try: next(gen)
            except StopIteration: break
            if k % n_sub == 0:
                x, y, z, v, a, h = ag.sample(); rows.append((ag.t, x, y, z, v, a, h, ag.acting, ag.phase))
            ag.step(); k += 1
        vis = [(math.hypot(r[1], r[2]) < min(MAX_RANGE, horizon(r[3]))) for r in rows]
        best = (0, 0); i = 0
        while i < len(vis):
            if vis[i]:
                j = i
                while j < len(vis) and vis[j]: j += 1
                if j - i > best[1] - best[0]: best = (i, j)
                i = j
            else: i += 1
        rows = rows[best[0]:best[1]]
        if len(rows) >= min_packets: break
    else:
        raise RuntimeError(f"could not build a visible {cls} track (seed {seed})")
    n = len(rows); t_rel = np.arange(n) * DT_OUT
    P = np.array([[r[1], r[2], r[3]] for r in rows]); V = np.array([r[4] for r in rows]); A = np.array([r[5] for r in rows])
    hd = np.degrees(np.array([r[6] for r in rows])) % 360; tt = t0 + t_rel
    df = pd.DataFrame({"packet_size": 166, "source": 7, "destination": 255, "packet_type": 163, "packet_category": 11, "radar_id": 1, "track_no": track_no,
        "pos_x": P[:, 0], "pos_y": P[:, 1], "pos_z": P[:, 2], "range_mag": np.linalg.norm(P, axis=1),
        "vel_x": V[:, 0], "vel_y": V[:, 1], "vel_z": V[:, 2], "vel_mag": np.linalg.norm(V, axis=1), "heading": hd,
        "accel_x": A[:, 0], "accel_y": A[:, 1], "accel_z": A[:, 2], "accel_mag": np.linalg.norm(A, axis=1)})
    names = ["std_dev_x", "std_dev_y", "std_dev_z", "std_dev_vx", "std_dev_vy", "std_dev_vz", "std_dev_ax", "std_dev_ay", "std_dev_az", "cov_x_vx", "cov_y_vy", "cov_z_vz"]
    if uncertainty:
        U = nominal_uncertainty(P, cls)
        for ax, c in enumerate("xyz"):
            df["std_dev_" + c] = np.sqrt(U[:, ax, 0, 0]); df["std_dev_v" + c] = np.sqrt(U[:, ax, 1, 1]); df["std_dev_a" + c] = np.sqrt(U[:, ax, 2, 2]); df["cov_%s_v%s" % (c, c)] = U[:, ax, 0, 1]
    else:
        for c in names: df[c] = 0.0
    df["ntp_sec"] = np.floor(tt).astype(np.int64); df["ntp_frac"] = ((tt % 1) * 1e6).round().astype(np.int64); df["assoc_indication"] = 1
    sp = np.linalg.norm(V, axis=1); vhat = V / np.maximum(sp, 1.0)[:, None]
    a_al = (A * vhat).sum(1); a_lat = np.linalg.norm(A - a_al[:, None] * vhat, axis=1)
    df["time_s"] = t_rel; df["true_class"] = cls; df["acting_class"] = [r[7] for r in rows]; df["phase"] = [r[8] for r in rows]
    df["behaviour_group"] = [PHASE_GROUP.get(p, "Other") for p in df["phase"]]
    df["subtype"] = meta["subtype"]; df["geometry"] = meta.get("geometry", ""); df["disguised"] = bool(meta.get("disguised", False))
    df["disguise_type"] = f"{cls}->{meta['mimic']}" if meta.get("disguised") else ""
    bo = np.nan
    if meta.get("disguised"):
        rev = (df["acting_class"] == cls).to_numpy(); mim = ~rev
        if mim.any() and rev.any() and mim.argmax() < rev.argmax(): bo = float(df["time_s"][rev & (np.arange(n) > np.where(mim)[0].max())].min()) if (rev & (np.arange(n) > np.where(mim)[0].max())).any() else np.nan
    df["breakout_time_s"] = bo
    df["speed"] = sp; df["lateral_g"] = a_lat / G
    return df, meta

CSV_COLS = ICD_COLS + TRUTH_COLS

def generate_dataset(packets_per_class=40000, seed=7000, p_disguise=0.45, uncertainty=True, round_decimals=3, progress=True, classes=None):
    """Balanced by PACKETS: every class gets (about) the same number of packets. Whole tracks only, so each class overshoots by < 1 track."""
    import time
    t0 = time.time(); frames = []; tn = 10000
    for ci, cls in enumerate(classes or CLASSES):
        got = 0; i = 0
        while got < packets_per_class:
            df, meta = simulate_track(cls, seed + ci * 100000 + i, tn, p_disguise=p_disguise, uncertainty=uncertainty); frames.append(df); got += len(df); i += 1; tn += 1
        if progress: print(f"{cls:18s} {i:5d} tracks  {got:7d} packets   ({time.time()-t0:.0f}s)")
    D = pd.concat(frames, ignore_index=True)[CSV_COLS]
    if round_decimals is not None:
        fl = D.select_dtypes("float").columns; D[fl] = D[fl].round(round_decimals)
    return D

## Part B: generate the data, save the CSV, explore it

In [ ]:
# ---------------- Cell 1: settings ----------------
import os, time, math, warnings, html
import numpy as np, pandas as pd
import plotly.graph_objects as go, plotly.io as pio
from plotly.subplots import make_subplots
warnings.filterwarnings("ignore")
import sys; S = sys.modules['__main__']   # the simulator cells above live in this notebook

PACKETS_PER_CLASS = 40000     # every class gets about this many packets (whole tracks, so it overshoots by < 1 track)
SEED = 7000                   # change for a different random dataset
P_DISGUISE = 0.45             # share of tracks that are disguised, for the classes that can be disguised
ADD_UNCERTAINTY = True        # fill std_dev_* / cov_* with NOMINAL tracker uncertainty (see report). False -> zeros
OUT_DIR = "radar_sim_v2_output"
CSV_NAME = "radar_sim_v2_dataset.csv"
ROUND_DECIMALS = 3
os.makedirs(OUT_DIR, exist_ok=True)

CLASSES = S.CLASSES
SHORT = {"CIVILIAN_AIRCRAFT":"Civilian","FIGHTER_AIRCRAFT":"Fighter","HELICOPTER":"Helicopter","DRONE":"Drone","SURFACE_MISSILE":"Surface missile",
         "AERIAL_MISSILE":"Aerial missile","SKIFF":"Skiff","BOAT":"Boat","SHIP":"Ship"}
COL = {"CIVILIAN_AIRCRAFT":"#4cc9f0","FIGHTER_AIRCRAFT":"#e63946","HELICOPTER":"#b388ff","DRONE":"#ffd166","SURFACE_MISSILE":"#f4a261",
       "AERIAL_MISSILE":"#ff70a6","SKIFF":"#80ed99","BOAT":"#2ec4b6","SHIP":"#adb5bd"}
MOTIONS = ["steady cruise","speed change","turning","climbing","descending","hover / near-stationary"]
MCOL = dict(zip(MOTIONS, ["#4c78a8","#f2cf5b","#e45756","#54a24b","#72b7b2","#b279a2"]))
BG, PANEL, GRID = "#0e1117", "#161b22", "#2a3140"
pio.templates.default = "plotly_dark"
FIGS = {}   # name -> figure (used by the report in the last cell)
def style(fig, h=700, title=None):
    fig.update_layout(template="plotly_dark", paper_bgcolor=BG, plot_bgcolor=PANEL, height=h, font=dict(family="Inter, Segoe UI, Arial", size=13, color="#e6edf3"),
                      title=dict(text=title, x=0.01, font=dict(size=20)) if title else None, legend=dict(bgcolor="rgba(0,0,0,0)"), margin=dict(l=60, r=30, t=90, b=50), hoverlabel=dict(bgcolor="#222b38"))
    fig.update_xaxes(gridcolor=GRID, zeroline=False); fig.update_yaxes(gridcolor=GRID, zeroline=False); return fig
def show(name, fig): FIGS[name] = fig; fig.show()

In [ ]:
# ---------------- Cell 2: generate the tracks and SAVE THE CSV ----------------
t0 = time.time()
D = S.generate_dataset(packets_per_class=PACKETS_PER_CLASS, seed=SEED, p_disguise=P_DISGUISE, uncertainty=ADD_UNCERTAINTY, round_decimals=ROUND_DECIMALS)
csv_path = os.path.join(OUT_DIR, CSV_NAME); D.to_csv(csv_path, index=False)
pd.DataFrame([(k, v[0], v[1]) for k, v in S.PHASE_INFO.items()], columns=["phase", "behaviour_group", "meaning"]).to_csv(os.path.join(OUT_DIR, "behaviour_glossary.csv"), index=False)
print(f"\nSaved {csv_path}   rows={len(D):,}  tracks={D.track_no.nunique():,}  columns={D.shape[1]}  size={os.path.getsize(csv_path)/1e6:.1f} MB   ({time.time()-t0:.0f}s)")

In [ ]:
# ---------------- Cell 3: load the CSV and add the motion label ----------------
D = pd.read_csv(os.path.join(OUT_DIR, CSV_NAME))
D = S.add_motion_state(D); D["alt"] = D.pos_z
cnt = D.groupby("true_class").agg(tracks=("track_no", "nunique"), packets=("track_no", "size")).reindex(CLASSES)
cnt["share_%"] = (cnt.packets / cnt.packets.sum() * 100).round(2); display(cnt)

## 1. The dataset at a glance

In [ ]:
# ---------------- Cell 4: dashboard ----------------
fig = make_subplots(rows=2, cols=3, specs=[[{}, {}, {}], [{}, {"type": "domain"}, {"type": "domain"}]], horizontal_spacing=.07, vertical_spacing=.2,
    subplot_titles=("Where each class lives (speed vs altitude)", "Tracks per class (labels = packets)", "Speed range per class", "How hard each class manoeuvres (99th pct sideways g)", "Disguised tracks", "What the targets are doing (behaviour groups)"))
for c in CLASSES:
    d = D[D.true_class == c]; s = d.sample(min(2500, len(d)), random_state=0)
    fig.add_trace(go.Scattergl(x=s.speed + .5, y=s.alt + 1, mode="markers", marker=dict(size=4, color=COL[c], opacity=.55), name=SHORT[c], legendgroup=c,
        text=s.phase, hovertemplate="%{text}<br>speed %{x:.0f} m/s<br>alt %{y:.0f} m<extra>" + SHORT[c] + "</extra>"), 1, 1)
    fig.add_trace(go.Bar(x=[SHORT[c]], y=[d.track_no.nunique()], marker_color=COL[c], legendgroup=c, showlegend=False, text=[f"{len(d):,}"], textposition="outside", hovertemplate="%{x}: %{y} tracks<extra></extra>"), 1, 2)
    q = d.speed.quantile([0, .05, .25, .5, .75, .95, 1]).values
    fig.add_trace(go.Box(x=[SHORT[c]], q1=[q[2]], median=[q[3]], q3=[q[4]], lowerfence=[q[1]], upperfence=[q[5]], mean=[d.speed.mean()], marker_color=COL[c], legendgroup=c, showlegend=False, name=SHORT[c]), 1, 3)
    fig.add_trace(go.Bar(x=[SHORT[c]], y=[d.lateral_g.quantile(.99)], marker_color=COL[c], legendgroup=c, showlegend=False, hovertemplate="%{x}: %{y:.2f} g<extra></extra>"), 2, 1)
fig.update_xaxes(type="log", title_text="speed (m/s)", row=1, col=1); fig.update_yaxes(type="log", title_text="altitude (m)", row=1, col=1)
fig.update_yaxes(type="log", title_text="speed (m/s)", row=1, col=3); fig.update_yaxes(title_text="sideways g (99th pct)", row=2, col=1)
dis = D[D.disguised].groupby("disguise_type").track_no.nunique().sort_values(ascending=False); cls_of = [t.split("->")[0] for t in dis.index]
fig.add_trace(go.Pie(labels=[t.replace("_AIRCRAFT", "").replace("->", " as ").title() for t in dis.index], values=dis.values, hole=.55, marker=dict(colors=[COL[c] for c in cls_of], line=dict(color=BG, width=2)), textinfo="label+value", showlegend=False, sort=False), 2, 2)
bg = D.behaviour_group.value_counts()
fig.add_trace(go.Pie(labels=bg.index, values=bg.values, hole=.55, textinfo="label+percent", showlegend=False, marker=dict(line=dict(color=BG, width=2))), 2, 3)
fig.add_annotation(text=f"{D.disguised.groupby(D.track_no).first().sum()} of {D.track_no.nunique():,}<br>tracks", x=.5, y=.17, xref="x domain", yref="y domain", showarrow=False) if False else None
style(fig, 900, "Radar simulator v2.1: the dataset at a glance (click a class in the legend to hide or isolate it)"); fig.update_layout(legend=dict(orientation="h", y=-.08, x=.0))
show("dashboard", fig)

## 2. What are the targets doing? (and does every class get to show every kind of motion?)

In [ ]:
# ---------------- Cell 5: motion composition + coverage ----------------
ms = pd.crosstab(D.true_class, D.motion, normalize="index").reindex(index=CLASSES, columns=MOTIONS).fillna(0) * 100
cov = pd.DataFrame({m: [(D[(D.true_class == c) & (D.motion == m)].groupby("track_no").size() >= 3).sum() / D[D.true_class == c].track_no.nunique() * 100 for c in CLASSES] for m in MOTIONS}, index=CLASSES)
fig = make_subplots(rows=1, cols=2, column_widths=[.52, .48], subplot_titles=("Share of packets in each motion state", "Coverage: % of tracks that contain each motion (3+ packets)"), horizontal_spacing=.14)
for m in MOTIONS: fig.add_trace(go.Bar(y=[SHORT[c] for c in CLASSES], x=ms[m], name=m, orientation="h", marker_color=MCOL[m], hovertemplate="%{y}: %{x:.1f}%<extra>" + m + "</extra>"), 1, 1)
fig.update_layout(barmode="stack"); fig.update_yaxes(autorange="reversed", row=1, col=1); fig.update_xaxes(title_text="% of packets", range=[0, 100], row=1, col=1)
fig.add_trace(go.Heatmap(z=cov.values, x=MOTIONS, y=[SHORT[c] for c in CLASSES], colorscale="Viridis", zmin=0, zmax=100, text=cov.round(0).astype(int).values, texttemplate="%{text}", colorbar=dict(title="% tracks", x=1.02), hovertemplate="%{y} / %{x}: %{z:.0f}% of tracks<extra></extra>"), 1, 2)
fig.update_yaxes(autorange="reversed", row=1, col=2); fig.update_xaxes(tickangle=30, row=1, col=2)
style(fig, 560, "Motion mix per class: nobody is only cruising, and rare motions still appear in many tracks"); fig.update_layout(legend=dict(orientation="h", y=-.2))
show("motion", fig)

## 3. Where the classes live, and how alike they are

In [ ]:
# ---------------- Cell 6: speed-altitude zones ----------------
fig = go.Figure(); xs, ys = np.log10(D.speed + 1), np.log10(D.alt + 1)
for c in CLASSES:
    m = (D.true_class == c).values; H, xe, ye = np.histogram2d(xs[m], ys[m], bins=50, range=[[0, 3.3], [0, 4.4]]); H = H / H.max()
    fig.add_trace(go.Contour(z=H.T, x=.5*(xe[1:] + xe[:-1]), y=.5*(ye[1:] + ye[:-1]), contours=dict(start=.07, end=.07, size=1, coloring="lines"), colorscale=[[0, COL[c]], [1, COL[c]]], line=dict(width=2.5), showscale=False, name=SHORT[c], showlegend=True, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=[np.median(xs[m])], y=[np.median(ys[m])], mode="text", text=[SHORT[c]], textfont=dict(color=COL[c], size=14), showlegend=False, hoverinfo="skip"))
fig.update_xaxes(tickvals=[0, 1, 2, 3], ticktext=["1", "10", "100", "1000"], title="speed (m/s)"); fig.update_yaxes(tickvals=[0, 1, 2, 3, 4], ticktext=["0", "10", "100", "1 000", "10 000"], title="altitude (m)")
style(fig, 650, "Behaviour zones: outlines contain where each class spends its time. Overlaps = classes motion alone cannot separate"); show("zones", fig)

In [ ]:
# ---------------- Cell 7: overlap heat-map ----------------
feat = np.column_stack([np.log10(D.speed + 1), np.log10(D.alt + 1), np.log10(D.lateral_g + 0.01), np.log10(D.vel_z.abs() + 1)])
edges = [np.linspace(feat[:, k].min(), feat[:, k].max(), 16) for k in range(4)]; P = {}
for c in CLASSES:
    m = (D.true_class == c).values; P[c] = np.mean([np.histogram(feat[m, k], bins=edges[k])[0] / m.sum() for k in range(4)], axis=0)
O = np.array([[np.minimum(P[a], P[b]).sum() for b in CLASSES] for a in CLASSES]); nm = [SHORT[c] for c in CLASSES]
fig = go.Figure(go.Heatmap(z=O, x=nm, y=nm, colorscale="Magma", zmin=0, zmax=1, text=np.round(O, 2), texttemplate="%{text}", colorbar=dict(title="overlap"), hovertemplate="%{y} vs %{x}: %{z:.2f}<extra></extra>"))
fig.update_yaxes(autorange="reversed"); style(fig, 640, "How alike are the classes? (speed, height, sideways g, climb rate; 1 = identical)"); show("overlap", fig)

In [ ]:
# ---------------- Cell 8: speed vs sideways g ----------------
fig = go.Figure()
for c in CLASSES:
    d = D[D.true_class == c]; s = d.sample(min(3000, len(d)), random_state=1)
    fig.add_trace(go.Scattergl(x=s.speed + .5, y=s.lateral_g + 1e-3, mode="markers", marker=dict(size=4, color=COL[c], opacity=.5), name=SHORT[c], text=s.phase + " | " + s.subtype, hovertemplate="%{text}<br>%{x:.0f} m/s, %{y:.3f} g<extra></extra>"))
fig.update_xaxes(type="log", title="speed (m/s)"); fig.update_yaxes(type="log", title="sideways acceleration (g)", range=[-3.1, 1.4])
for g, t in [(.1, "gentle"), (1, "1 g"), (5, "hard")]: fig.add_hline(y=g, line=dict(color="#6e7681", dash="dot"), annotation_text=t, annotation_position="bottom right")
style(fig, 650, "Speed vs manoeuvring: what each class can physically do"); show("speed_g", fig)

## 4. What behaviours does the generator produce?

In [ ]:
# ---------------- Cell 9: sunburst (class -> behaviour group -> phase) ----------------
g = D.groupby(["true_class", "behaviour_group", "phase"]).size().reset_index(name="n"); ids, labels, parents, values, colors = [], [], [], [], []
for c in CLASSES:
    sub = g[g.true_class == c]; ids.append(c); labels.append(SHORT[c]); parents.append(""); values.append(int(sub.n.sum())); colors.append(COL[c])
    for bgp, s2 in sub.groupby("behaviour_group"):
        ids.append(f"{c}/{bgp}"); labels.append(bgp); parents.append(c); values.append(int(s2.n.sum())); colors.append(COL[c])
        for _, r in s2.iterrows(): ids.append(f"{c}/{bgp}/{r.phase}"); labels.append(r.phase); parents.append(f"{c}/{bgp}"); values.append(int(r.n)); colors.append(COL[c])
fig = go.Figure(go.Sunburst(ids=ids, labels=labels, parents=parents, values=values, branchvalues="total", marker=dict(colors=colors, line=dict(color=BG, width=1)), hovertemplate="%{label}: %{value:,} packets (%{percentRoot:.1%} of class)<extra></extra>", maxdepth=3, insidetextorientation="radial"))
style(fig, 800, "Behaviour map: class (centre) -> behaviour group -> phase. Click a slice to zoom in"); show("sunburst", fig)

In [ ]:
# ---------------- Cell 10: behaviour groups per class ----------------
bgm = pd.crosstab(D.true_class, D.behaviour_group, normalize="index").reindex(CLASSES).fillna(0) * 100
fig = go.Figure()
for gname in bgm.columns: fig.add_trace(go.Bar(y=[SHORT[c] for c in CLASSES], x=bgm[gname], name=gname, orientation="h", hovertemplate="%{y}: %{x:.1f}%<extra>" + gname + "</extra>"))
fig.update_layout(barmode="stack"); fig.update_yaxes(autorange="reversed"); fig.update_xaxes(title="% of packets", range=[0, 100])
style(fig, 560, "Behaviour groups per class"); fig.update_layout(legend=dict(orientation="h", y=-.2)); show("groups", fig)

## 5. Example tracks (pick a class from the drop-down)

In [ ]:
# ---------------- Cell 11: track explorer ----------------
fig = make_subplots(rows=2, cols=2, subplot_titles=("Top view (km)", "Speed (m/s)", "Altitude (m)", "Sideways g"), horizontal_spacing=.1, vertical_spacing=.16)
PAL = ["#4cc9f0", "#ffd166", "#ff70a6", "#80ed99"]; groups = {}
for c in CLASSES:
    d = D[D.true_class == c]; L = d.groupby("track_no").size().sort_values(); tns = [L.index[int(q * (len(L) - 1))] for q in (.15, .4, .65, .9)]; groups[c] = []
    for k, tn in enumerate(tns):
        g = d[d.track_no == tn]; nm = f"{g.subtype.iloc[0]} ({len(g)} pkts)"; txt = g.phase + " | " + g.acting_class.map(SHORT)
        for (r, cc, y) in [(1, 1, None), (1, 2, g.speed), (2, 1, g.pos_z), (2, 2, g.lateral_g)]:
            if y is None: tr = go.Scatter(x=g.pos_x / 1e3, y=g.pos_y / 1e3, mode="lines+markers", marker=dict(size=4), line=dict(width=1.5, color=PAL[k]), name=nm, legendgroup=f"{c}{k}", text=txt, hovertemplate="%{text}<br>x %{x:.1f} km, y %{y:.1f} km<extra></extra>", visible=(c == CLASSES[0]))
            else: tr = go.Scatter(x=g.time_s, y=y, mode="lines", line=dict(width=1.6, color=PAL[k]), name=nm, legendgroup=f"{c}{k}", showlegend=False, text=txt, hovertemplate="%{text}<br>t=%{x:.0f}s %{y:.2f}<extra></extra>", visible=(c == CLASSES[0]))
            fig.add_trace(tr, r, cc); groups[c].append(len(fig.data) - 1)
buttons = []
for c in CLASSES:
    vis = [False] * len(fig.data)
    for i in groups[c]: vis[i] = True
    buttons.append(dict(label=SHORT[c], method="update", args=[{"visible": vis}, {"title.text": f"Example tracks: {SHORT[c]} (short, medium, long, longest)"}]))
fig.update_layout(updatemenus=[dict(buttons=buttons, direction="down", x=.0, xanchor="left", y=1.18, bgcolor="#222b38", font=dict(color="#e6edf3"))])
fig.update_yaxes(scaleanchor="x", scaleratio=1, row=1, col=1); fig.update_xaxes(title_text="time (s)", row=1, col=2); fig.update_xaxes(title_text="time (s)", row=2, col=1); fig.update_xaxes(title_text="time (s)", row=2, col=2)
style(fig, 800, f"Example tracks: {SHORT[CLASSES[0]]} (short, medium, long, longest)"); show("explorer", fig)

## 6. Hovering (helicopters and multirotor drones)

In [ ]:
# ---------------- Cell 12: hovering ----------------
def phase_pie(c):
    ph = D[D.true_class == c].phase.value_counts(); small = ph[ph / ph.sum() < .03]
    return pd.concat([ph[ph / ph.sum() >= .03], pd.Series({"other": small.sum()})]) if len(small) else ph
def longest_hover(cls, sub):
    d = D[(D.true_class == cls) & D.subtype.str.startswith(sub)]; best = (0, None, 0)
    for tn, g in d.groupby("track_no"):
        h = (g.speed.values < 1.5); run = 0; mx = 0; end = 0
        for i, v in enumerate(h):
            run = run + 1 if v else 0
            if run > mx: mx, end = run, i
        if mx > best[0]: best = (mx, tn, end - mx + 1)
    return best
fig = make_subplots(rows=2, cols=3, specs=[[{"type": "domain"}, {"type": "domain"}, {}], [{}, {}, {}]], vertical_spacing=.18, horizontal_spacing=.08,
    subplot_titles=("Helicopter: time spent per phase", "Drone: time spent per phase", "Share of packets at (almost) zero speed", "Helicopter: longest hover (colour = time)", "Multirotor drone: longest hover", "Helicopter speed through that track"))
for k, c in enumerate(["HELICOPTER", "DRONE"]):
    ph = phase_pie(c); fig.add_trace(go.Pie(labels=ph.index, values=ph.values, hole=.5, textinfo="label+percent", showlegend=False, marker=dict(line=dict(color=BG, width=2))), 1, k + 1)
hv = D.assign(h=D.speed < 1.5).groupby("true_class").h.mean().reindex(CLASSES) * 100
fig.add_trace(go.Bar(x=[SHORT[c] for c in CLASSES], y=hv.values, marker_color=[COL[c] for c in CLASSES], text=[f"{v:.0f}%" for v in hv], textposition="outside", showlegend=False, hovertemplate="%{x}: %{y:.1f}%<extra></extra>"), 1, 3)
for k, (cls, sub) in enumerate([("HELICOPTER", "observation"), ("DRONE", "multirotor")]):
    n, tn, s0 = longest_hover(cls, sub); g = D[D.track_no == tn].iloc[s0:s0 + n]; xx, yy = g.pos_x - g.pos_x.mean(), g.pos_y - g.pos_y.mean()
    fig.add_trace(go.Scatter(x=xx, y=yy, mode="lines+markers", line=dict(color="#6e7681", width=1), marker=dict(size=7, color=g.time_s, colorscale="Plasma", showscale=False), showlegend=False,
        hovertemplate="t=%{marker.color:.0f}s<br>%{x:.2f}, %{y:.2f} m<extra></extra>"), 2, k + 1)
    fig.update_xaxes(title_text=f"east (m)   hover lasts {n*2.5:.0f} s, wanders {np.ptp(xx):.1f} x {np.ptp(yy):.1f} m", row=2, col=k + 1); fig.update_yaxes(title_text="north (m)", row=2, col=k + 1)
n, tn, s0 = longest_hover("HELICOPTER", "observation"); g = D[D.track_no == tn]
fig.add_trace(go.Scatter(x=g.time_s, y=g.speed, mode="lines", line=dict(color=COL["HELICOPTER"]), showlegend=False), 2, 3)
fig.add_shape(type="rect", xref="x4", yref="y4 domain", x0=g.time_s.iloc[s0], x1=g.time_s.iloc[s0 + n - 1], y0=0, y1=1, fillcolor="#b279a2", opacity=.25, line_width=0)
fig.update_xaxes(title_text="time (s)", row=2, col=3); fig.update_yaxes(title_text="speed (m/s)", row=2, col=3); fig.update_yaxes(title_text="% of packets", row=1, col=3)
style(fig, 800, "Hovering: it appears only where it physically can (helicopters, multirotor drones); vessels just drift slowly"); show("hover", fig)

## 7. Disguises

In [ ]:
# ---------------- Cell 13: disguises ----------------
dd = D[D.disguised]; pairs = sorted(dd.disguise_type.unique()); ex = []
for p in pairs:
    for tn in dd[dd.disguise_type == p].track_no.unique()[:2]: ex.append((p, tn))
fig = go.Figure()
for i, (p, tn) in enumerate(ex):
    g = D[D.track_no == tn]; c = g.true_class.iloc[0]; mim = (g.acting_class != c).values; bo = g.breakout_time_s.iloc[0]; vis = (i == 0); lab = p.replace("->", " acting as ")
    fig.add_trace(go.Scatter(x=g.time_s[mim], y=g.speed[mim], mode="lines", line=dict(color="#ffb703", width=3), name="disguised", visible=vis, text=g.phase[mim], hovertemplate="%{text}<br>t=%{x:.0f}s  %{y:.1f} m/s<extra>disguised</extra>"))
    fig.add_trace(go.Scatter(x=g.time_s[~mim], y=g.speed[~mim], mode="lines", line=dict(color=COL[c], width=3), name="revealed (true behaviour)", visible=vis, text=g.phase[~mim], hovertemplate="%{text}<br>t=%{x:.0f}s  %{y:.1f} m/s<extra>revealed</extra>"))
    fig.add_trace(go.Scatter(x=[bo, bo], y=[0, g.speed.max() * 1.05], mode="lines", line=dict(color="#ffffff", dash="dash", width=1), name="breakout", visible=vis, hoverinfo="skip"))
btn = []
for i, (p, tn) in enumerate(ex):
    v = [False] * len(fig.data); v[3 * i:3 * i + 3] = [True] * 3; btn.append(dict(label=f"{p.replace('->', ' as ').replace('_AIRCRAFT', '')} #{tn}", method="update", args=[{"visible": v}]))
fig.update_layout(updatemenus=[dict(buttons=btn, direction="down", x=0, xanchor="left", y=1.14, bgcolor="#222b38", font=dict(color="#e6edf3"))])
fig.update_xaxes(title="time (s)"); fig.update_yaxes(title="speed (m/s)"); style(fig, 560, "A disguised target: orange = behaving like another class, coloured = true behaviour after the breakout (pick an example)"); show("disguise_examples", fig)

In [ ]:
# ---------------- Cell 14: disguise statistics ----------------
pt = dd.assign(dis=dd.acting_class != dd.true_class).groupby("disguise_type").dis.mean() * 100; cnt = dd.groupby("disguise_type").track_no.nunique(); bo = dd.groupby("track_no").agg(b=("breakout_time_s", "first"), p=("disguise_type", "first"))
fig = make_subplots(rows=1, cols=3, subplot_titles=("Disguised tracks per pair", "Share of packets disguised vs revealed", "When does the breakout happen? (s from track start)"), horizontal_spacing=.09)
lab = [p.replace("_AIRCRAFT", "").replace("->", " as ").title() for p in cnt.index]
fig.add_trace(go.Bar(x=lab, y=cnt.values, marker_color=[COL[p.split("->")[0]] for p in cnt.index], showlegend=False, text=cnt.values, textposition="outside"), 1, 1)
fig.add_trace(go.Bar(x=lab, y=pt.values, name="disguised", marker_color="#ffb703"), 1, 2); fig.add_trace(go.Bar(x=lab, y=100 - pt.values, name="revealed", marker_color="#4c78a8"), 1, 2); fig.update_layout(barmode="stack")
for p in cnt.index:
    h, e = np.histogram(bo[bo.p == p].b, bins=np.linspace(0, bo.b.max(), 16)); fig.add_trace(go.Scatter(x=.5 * (e[1:] + e[:-1]), y=h, mode="lines+markers", line=dict(shape="spline", color=COL[p.split("->")[0]], dash="solid" if p.endswith(("CIVILIAN_AIRCRAFT", "BOAT", "SHIP")) else "dot"), name=p.replace("_AIRCRAFT", "").replace("->", " as ").title()), 1, 3)
fig.update_layout(barmode="stack"); fig.update_yaxes(title_text="tracks", row=1, col=3); fig.update_xaxes(tickangle=25, row=1, col=1); fig.update_xaxes(tickangle=25, row=1, col=2); fig.update_yaxes(title_text="% of packets", row=1, col=2)
style(fig, 520, f"Disguises: {dd.track_no.nunique()} of {D.track_no.nunique():,} tracks ({dd.track_no.nunique()/D[D.true_class.isin(['FIGHTER_AIRCRAFT','DRONE','SKIFF','BOAT'])].track_no.nunique():.0%} of the four classes that can be disguised)"); show("disguise_stats", fig)

## 8. Data health checks

In [ ]:
# ---------------- Cell 15: health ----------------
dt = D.groupby("track_no").time_s.diff().dropna(); L = D.groupby(["true_class", "track_no"]).size().reset_index(name="n"); err = []
for tn, g in D.groupby("track_no"):
    p = g[["pos_x", "pos_y", "pos_z"]].to_numpy(); v = g[["vel_x", "vel_y", "vel_z"]].to_numpy(); vm = .5 * (v[1:] + v[:-1])
    err.append((g.true_class.iloc[0], np.median(np.linalg.norm(np.diff(p, axis=0) / 2.5 - vm, axis=1) / np.maximum(np.linalg.norm(vm, axis=1), 1)) * 100))
E = pd.DataFrame(err, columns=["c", "e"])
fig = make_subplots(rows=2, cols=2, subplot_titles=(f"Time step between packets ({dt.min():.1f} to {dt.max():.1f} s)", "Packets per track (x 2.5 s = duration)", "Velocity vs position change: median mismatch (%)", "Nominal position uncertainty (std_dev_x) vs range"), vertical_spacing=.2, horizontal_spacing=.1)
fig.add_trace(go.Histogram(x=dt, xbins=dict(start=2.4, end=2.6, size=.05), marker_color="#2ec4b6", showlegend=False), 1, 1)
for c in CLASSES:
    q = L[L.true_class == c].n; fig.add_trace(go.Box(y=q, name=SHORT[c], marker_color=COL[c], showlegend=False), 1, 2)
    fig.add_trace(go.Bar(x=[SHORT[c]], y=[E[E.c == c].e.median()], marker_color=COL[c], showlegend=False), 2, 1)
    d = D[D.true_class == c].sample(min(1500, (D.true_class == c).sum()), random_state=2); fig.add_trace(go.Scattergl(x=d.range_mag / 1e3, y=d.std_dev_x, mode="markers", marker=dict(size=3, color=COL[c], opacity=.5), name=SHORT[c], legendgroup=c), 2, 2)
fig.update_xaxes(title_text="range (km)", row=2, col=2); fig.update_yaxes(title_text="std_dev_x (m)", type="log", row=2, col=2); fig.update_yaxes(title_text="% mismatch", row=2, col=1)
style(fig, 800, "Data health: exact 2.5 s grid, track lengths, physical consistency, nominal uncertainty"); show("health", fig)

## 9. Glossary: what do the behaviour names mean?

In [ ]:
# ---------------- Cell 16a: glossaries (shown here and in the report) ----------------
SUBTYPE_INFO = {
 "airliner": "Large passenger jet: 200-255 m/s at 9-12.5 km, banks up to about 0.3-0.5 g.", "turboprop": "Regional propeller airliner: 105-150 m/s at 3-7 km.", "light_ga": "Light general-aviation aircraft: 45-75 m/s, low altitude, tighter turns, many circuits.",
 "cap": "Fighter combat air patrol: long race-track at height with short sprints.", "high_speed": "Fighter flying fast legs (420-750 m/s) with hard turns between them.", "high_speed_disguised": "Same, but first pretends to be an airliner, then breaks out.",
 "bfm": "Basic fighter manoeuvres: repeated hard turns, zoom climbs and dives, speed bleeding off.", "low_level": "Fighter flying low (80-300 m) with jinks.", "climb_cycles": "Low flying with climbs up and descents back down.", "transit": "Straight-ish flight between points.",
 "observation": "Helicopter that hovers, repositions and hovers again.", "search": "Helicopter circling and travelling while searching an area.", "shuttle": "Helicopter hover-taxiing short hops near the ground.", "approach_land": "Helicopter slowing, hovering and descending vertically to land.",
 "multirotor": "Small quad/hex drone: hovers, stop-and-go, survey lines, small circles, up to about 30 m/s.", "fixed_wing": "Small fixed-wing drone: loiters, survey legs, waypoints, 20-45 m/s.", "male_hale": "Large high-altitude endurance drone: slow, very wide turns, 4-15 km.",
 "loitering_uav": "Medium-speed drone that loiters, transits, changes height and dashes.", "loitering_uav_disguised": "Same, but first behaves like a helicopter or an airliner, then breaks out.",
 "sea_skimming_cruise": "Low (8-25 m) steady-speed powered flight over the sea, with course, speed and height changes.", "supersonic_cruise": "Same but 600-950 m/s.", "terrain_following_cruise": "Low flight with a rolling height profile.", "launch_climb_cruise": "Boost and steep climb after launch, then level cruise.", "high_altitude_cruise": "Cruise at 3-9 km with changes.",
 "short_range_boost_coast": "Short motor burn then coasting (speed decays by drag), small course corrections.", "medium_range_boost_sustain_coast": "Boost, a lower-thrust sustain phase, then coast.", "vertical_launch_pitchover": "Launched upwards, then leans over onto a horizontal heading.", "long_range_cruise": "Boost then a long powered cruise at 550-900 m/s at height.",
 "transit_disguised": "Vessel that first behaves like a larger or smaller class and then breaks out; the name is the vessel's first behaviour."}
print("Subtype note: for skiffs, boats and ships the subtype is the vessel's FIRST behaviour; it then switches between its behaviours (see 'phase').")
gl = pd.DataFrame([(k, v[0], v[1]) for k, v in S.PHASE_INFO.items()], columns=["phase", "behaviour group", "meaning"]); display(gl)

In [ ]:
# ---------------- Cell 16b: write ONE self-contained interactive HTML report ----------------
def tbl(df):
    return "<table>" + "<tr>" + "".join(f"<th>{html.escape(str(c))}</th>" for c in df.columns) + "</tr>" + "".join("<tr>" + "".join(f"<td>{html.escape(str(v))}</td>" for v in r) + "</tr>" for r in df.itertuples(index=False)) + "</table>"
def fig_html(name): return FIGS[name].to_html(full_html=False, include_plotlyjs=False, config=dict(displaylogo=False, responsive=True))
nd = D.track_no.nunique(); elig = D[D.true_class.isin(["FIGHTER_AIRCRAFT", "DRONE", "SKIFF", "BOAT"])].track_no.nunique(); ndis = D[D.disguised].track_no.nunique()
cs = D.groupby("true_class").agg(tracks=("track_no", "nunique"), packets=("track_no", "size")).reindex(CLASSES); cs["share"] = (cs.packets / cs.packets.sum() * 100).round(2).astype(str) + " %"; cs.index = [SHORT[c] for c in cs.index]
gl_html = tbl(gl.groupby("behaviour group", sort=False).apply(lambda x: x).reset_index(drop=True))
sub_df = pd.DataFrame([(k, v) for k, v in SUBTYPE_INFO.items()], columns=["subtype", "meaning"])
CSS = "body{background:#0e1117;color:#e6edf3;font-family:Inter,Segoe UI,Arial,sans-serif;margin:0;line-height:1.55}.wrap{max-width:1250px;margin:0 auto;padding:30px 24px}h1{font-size:30px;margin:0 0 6px}h2{margin-top:46px;border-bottom:1px solid #2a3140;padding-bottom:8px}.card{background:#161b22;border:1px solid #2a3140;border-radius:12px;padding:16px 20px;margin:14px 0}.kpi{display:flex;gap:14px;flex-wrap:wrap}.kpi div{background:#161b22;border:1px solid #2a3140;border-radius:12px;padding:12px 18px;min-width:150px}.kpi b{display:block;font-size:24px;color:#4cc9f0}table{border-collapse:collapse;width:100%;font-size:13px;margin:10px 0}th{background:#1f2733;text-align:left}td,th{border:1px solid #2a3140;padding:6px 10px;vertical-align:top}code{background:#1f2733;padding:1px 6px;border-radius:4px}.note{color:#9fb0c3}.warn{border-left:4px solid #ffb703}"
parts = [f"<!doctype html><html><head><meta charset='utf-8'><title>Radar simulator v2.1 report</title><style>{CSS}</style><script>{__import__("plotly.offline",fromlist=["x"]).get_plotlyjs()}</script></head><body><div class='wrap'>",
 "<h1>Radar simulator v2.1: data report</h1><div class='note'>Noise-free truth sampled every 2.5 s. Interactive: hover, zoom, click legends, use the drop-downs.</div>",
 f"<div class='kpi'><div><b>{len(D):,}</b>packets</div><div><b>{nd:,}</b>tracks</div><div><b>9</b>classes</div><div><b>2.5 s</b>time step</div><div><b>{ndis}</b>disguised tracks ({ndis/elig:.0%} of the 4 classes that can be disguised)</div></div>",
 "<h2>What changed in this version</h2><div class='card'><ul>"
 "<li><b>Balanced classes:</b> every class has about the same number of packets (table below). The number of tracks differs because tracks have different durations; short-lived targets such as missiles need more tracks to reach the same packet count.</li>"
 "<li><b>More manoeuvring, less cruising:</b> every class re-draws speed, heading and height often, and includes its full repertoire of manoeuvres (see the coverage map). Steady cruise is now a minority for aircraft, drones and missiles; vessels are naturally steadier.</li>"
 f"<li><b>More disguises:</b> {P_DISGUISE:.0%} of tracks in the four classes that can be disguised (fighter, drone, skiff, boat), with six pairs (fighter as civilian; drone as helicopter or civilian; skiff as boat; boat as ship or skiff), and abrupt or gradual breakouts.</li>"
 "<li><b>Missiles:</b> richer generic powered-flight profiles (boost, sustain, coast, cruise legs, course, speed and height changes, vertical launch). They contain no guidance, homing, target or evasion logic. For high-fidelity missile tracks, use data from an approved source.</li>"
 "<li><b>Uncertainty columns:</b> std_dev_* and cov_* are filled with nominal tracker values (explained below). They are reference columns and should not be used as classifier inputs.</li>"
 "<li><b>No split column:</b> split by track later (never by packet).</li></ul></div>",
 "<h2>Class balance</h2>", tbl(cs.reset_index().rename(columns={"index": "class"})), "<h2>1. The dataset at a glance</h2>", fig_html("dashboard"),
 "<h2>2. Motion mix and coverage</h2><div class='card'>Left: how each class spends its packets. Right: the share of tracks that contain each motion at least briefly. A class can be mostly cruising by time and still show every motion in most of its tracks; this is what lets the model learn rare manoeuvres.</div>", fig_html("motion"),
 "<h2>3. Zones and overlaps</h2>", fig_html("zones"), fig_html("overlap"), fig_html("speed_g"),
 "<h2>4. Behaviours</h2>", fig_html("sunburst"), fig_html("groups"),
 "<h2>5. Example tracks</h2>", fig_html("explorer"), "<h2>6. Hovering</h2>", fig_html("hover"), "<h2>7. Disguises</h2>", fig_html("disguise_examples"), fig_html("disguise_stats"), "<h2>8. Data health</h2>", fig_html("health"),
 "<h2>What the behaviour names mean</h2><div class='card warn'><b>Should the names be in the data?</b> Yes, as <b>ground-truth annotations</b> (columns <code>phase</code>, <code>behaviour_group</code>, <code>subtype</code>, <code>acting_class</code>) for analysis, validation and error diagnosis. They are never model inputs: a real radar does not tell you what the target is doing, so training on them would be cheating. Keep the vocabulary small and documented, as below.</div>",
 gl_html, "<h2>Subtypes</h2>", tbl(sub_df),
 "<h2>Design notes</h2><div class='card'><p><b>Realistic envelope, not realistic frequency.</b> A civilian aircraft rarely turns hard in real life, but the classifier must still see how a civilian turns, otherwise any hard turn would be labelled non-civilian. So each class gets its full repertoire in healthy proportions. If real-world frequencies matter for deployment, correct for them afterwards by re-weighting the classes (prior correction) instead of starving the model of rare motions.</p>"
 "<p><b>Uncertainty (std_dev, cov).</b> They describe the tracker, not the target, and depend on the real radar's accuracy, which we do not know. Zeros would mean 'perfect knowledge', which is wrong, so the columns hold nominal values from a covariance-only Kalman recursion (range sigma 25 m, azimuth and elevation 1 mrad, per-class process noise). They grow with range and are large on the first packets, as a real tracker's would. Replace them when the real radar parameters are known.</p>"
 "<p><b>Limits.</b> Class limits and behaviour mixes are engineering assumptions. Wind is a constant field per track. There is no terrain, coastline, traffic route or formation behaviour. Vertical manoeuvres stay below about 65 degrees of flight-path angle.</p></div>",
 "</div></body></html>"]
rep = os.path.join(OUT_DIR, "radar_sim_v2_report.html"); open(rep, "w", encoding="utf-8").write("".join(parts)); print("Report written:", os.path.abspath(rep), f"({os.path.getsize(rep)/1e6:.1f} MB)")